# 머신러닝-1 · 데이터 분할과 교차검증

**한 줄 요약** · 무엇을 어느 데이터로 정하는가 — 분할 → 교차검증 → 설정 탐색 → test 1회를 붓꽃 · 펭귄으로 끝까지

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 학습한 데이터로 채점한 점수가 실력이 아닌 이유를 엉터리 정답 실험(0.993)으로 설명한다
2. `train_test_split(stratify=, random_state=)`로 나누고, 분할 seed만 바꿔도 test 점수가 흔들림을 보인다
3. `StratifiedKFold`로 5-Fold를 직접 돌리고 `cross_val_score` · `cross_validate`와 같은 값을 낸다
4. `GridSearchCV` · `RandomizedSearchCV`로 설정을 고르고, 학습 · 검증 점수 차이로 과적합을 판단한다
5. train · val · test 3분할을 만들고, test로 고르면 점수가 부풀려지는 이유(0.467 vs 0.333)를 말한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **모델 점수가 99%로 나와 기뻐하기 전에** · 그 점수를 어느 데이터로 냈는지 확인 (1장)
- **이미지 · 표 데이터를 학습 · 시험용으로 나눌 때** · `stratify` · `random_state` (2장)
- **모델 두 개 중 무엇이 나은지 정할 때** · 같은 5등분의 CV mean ± std (3장)
- **깊이 · 이웃 수 · 학습률 같은 설정을 고를 때** · GridSearchCV · RandomizedSearchCV (4장)
- **딥러닝 프로젝트에서 데이터 폴더를 나눌 때** · train · val · test 3분할, test는 마지막 1회 (5장)

## 학습 지도

<img src="images/ml1_map.png" alt="머신러닝-1 학습 지도" width="837">

**예상 학습 시간** · 개념 1.5시간 + 실습 1시간 (전체 실행 1분 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">0</td><td>준비</td><td>라이브러리 · seed · 붓꽃 데이터</td></tr>
<tr><td style="text-align: center;">1</td><td>왜 나누는가</td><td>학습한 데이터로 낸 점수는 왜 믿을 수 없는가?</td></tr>
<tr><td style="text-align: center;">2</td><td>데이터 나누기</td><td>train · test는 어떻게 나누고, 한 번 나눈 점수에는 무엇이 섞이는가?</td></tr>
<tr><td style="text-align: center;">3</td><td>교차검증</td><td>여러 번 나눠 평균을 내는 방법과 도구 4가지는 어떻게 다른가?</td></tr>
<tr><td style="text-align: center;">4</td><td>하이퍼파라미터 탐색</td><td>설정값과 과적합은 어떤 관계이고, 후보를 자동으로 비교하려면?</td></tr>
<tr><td style="text-align: center;">5</td><td>최종 테스트 · 딥러닝 연결</td><td>검증 데이터와 최종 테스트는 어떻게 다르며, 3분할은 언제 쓰나?</td></tr>
<tr><td style="text-align: center;">6</td><td>종합 실습</td><td>펭귄으로 분할 → 비교 → 탐색 → test 1회를 처음부터 끝까지</td></tr>
<tr><td style="text-align: center;">—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

모든 장에서 쓰는 라이브러리와 붓꽃 데이터를 불러옴. 붓꽃은 sklearn 안에 들어 있어 파일이 필요 없고, 6장 펭귄은 seaborn 예제 데이터로 불러옴 (2-1 실습과 같은 방법).

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">데이터</th><th style="text-align: left;">내용</th><th style="text-align: left;">처음 나오는 곳</th></tr>
</thead>
<tbody>
<tr><td><code>X_full</code> · <code>y_full</code></td><td>붓꽃 150송이 × 특성 4개 · 정답 0 · 1 · 2 (클래스별 50송이, <strong>정답 순서로 정렬됨</strong>)</td><td>0</td></tr>
<tr><td><code>y_shuffled</code></td><td>정답을 무작위로 섞은 엉터리 정답 (배울 규칙이 없음)</td><td>1-2</td></tr>
<tr><td><code>X_train</code> · <code>X_test</code></td><td>8 : 2로 나눈 120송이 · 30송이</td><td>2-1</td></tr>
<tr><td><code>skf</code></td><td>섞은 5등분 규칙 (모든 비교에 같은 분할 사용)</td><td>3-1</td></tr>
<tr><td><code>sns.load_dataset("penguins")</code></td><td>펭귄 344마리 · 측정값 4개 · 품종 3개</td><td>6</td></tr>
</tbody></table>

**실행 흐름**

1. 라이브러리 불러오기
2. seed 고정 · 버전 기록
3. 붓꽃 불러오기 · 모양 확인
4. 정답이 정렬된 순서 확인

In [1]:
# 라이브러리 불러오기
# 왜 · 이 노트북 전체에서 쓰는 도구를 한곳에 모아 두기 위해
import sklearn                                       # scikit-learn = 머신러닝 표준 라이브러리
import numpy as np                                   # 배열 · 평균 · 표준편차
import pandas as pd                                  # 결과를 표로 정리
from sklearn.datasets import load_iris               # 붓꽃 150송이 (파일 필요 없음)
from sklearn.tree import DecisionTreeClassifier      # 질문으로 갈라 나가는 모델
from sklearn.linear_model import LogisticRegression  # 클래스별 확률 모델
from sklearn.neighbors import KNeighborsClassifier   # 이웃 k개 다수결 모델
from sklearn.metrics import accuracy_score           # 정답 비율


# 결과 고정용 seed (원본과 같은 18) · 버전 기록
# 왜 · 원본과 같은 seed로 숫자를 맞추고, 버전 차이로 달라질 수 있어 기록해 두기 위해
SEED = 18                           # 원본과 같은 seed → 같은 환경에서 같은 결과
print("sklearn:", sklearn.__version__, "| numpy:", np.__version__,  # 버전 기록 (재현용)
      "| pandas:", pd.__version__)  # 버전이 다르면 숫자가 조금 다를 수 있음


# 데이터 불러오기: X = 특성(문제), y = 정답
# 왜 · 문제와 정답을 나눠 두어 뒤의 모든 셀이 같은 이름을 쓰게 하기 위해
iris = load_iris()                               # sklearn 안에 든 데이터 → 파일 필요 없음
X_full, y_full = iris.data, iris.target          # (150, 4) 실수 · (150,) 정수
print("X:", X_full.shape, "| y:", y_full.shape)  # 문제 · 정답의 모양 확인
print("특성:", iris.feature_names)               # 열 4개의 이름
print("클래스:", iris.target_names.tolist(),     # 정답 이름과 클래스별 개수
      "| 클래스별 개수:", np.bincount(y_full))   # bincount → 0 · 1 · 2 개수


# 정답이 놓인 순서 확인 (3-2에서 중요)
# 왜 · 정답이 정렬돼 있으면 섞지 않은 분할이 위험하다는 것을 미리 알기 위해
print("y 앞 5개:", y_full[:5], "| y 뒤 5개:", y_full[-5:])  # 0 → 1 → 2 정렬

sklearn: 1.9.0 | numpy: 2.4.6 | pandas: 3.0.3
X: (150, 4) | y: (150,)
특성: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
클래스: ['setosa', 'versicolor', 'virginica'] | 클래스별 개수: [50 50 50]
y 앞 5개: [0 0 0 0 0] | y 뒤 5개: [2 2 2 2 2]


---
## 1. 왜 나누는가

> **이 장의 흐름**
>
> - Python · NumPy · Pandas로 데이터를 다룰 수 있게 됐다
> - 이제 데이터로 규칙을 배우는 모델을 씀.
> - 사용법은 3줄이지만, 점수를 어디서 재느냐에 따라 진짜와 가짜가 갈림

**파이프라인 위치** · ④ 모델 · ⑤ 학습 — 모델 만들기, `fit`, 점수 확인

### 머신러닝 종류 · 용어 <sub>원본 01 intro</sub>

머신러닝 = 사람이 규칙을 코딩하지 않고 **데이터로 규칙을 학습**해 새 데이터를 예측하는 방법 (예: 스팸 필터 · 추천 · 질병 진단 보조 · 자율주행 인식)

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">종류</th><th style="text-align: left;">정답(y)</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>지도학습 · 분류</td><td>있음 (라벨)</td><td>스팸/정상, 품종 맞히기</td><td>붓꽃 · 펭귄 (전부 분류)</td></tr>
<tr><td>지도학습 · 회귀</td><td>있음 (숫자)</td><td>집값 · 매출 예측</td><td>— (<code>stratify</code> 대신 <code>KFold</code>, 3-2 표)</td></tr>
<tr><td>비지도학습</td><td>없음</td><td>군집화(고객 묶기) · 차원 축소(PCA)</td><td>—</td></tr>
<tr><td>강화학습</td><td>보상</td><td>게임 · 로봇의 행동 선택</td><td>—</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">용어</th><th style="text-align: left;">뜻</th><th style="text-align: left;">처음 나오는 곳</th></tr>
</thead>
<tbody>
<tr><td>Feature (특성, <code>X</code>)</td><td>모델이 입력으로 받는 정보 (꽃잎 길이 등)</td><td>0</td></tr>
<tr><td>Label · Target (정답, <code>y</code>)</td><td>모델이 맞히려는 값 (품종)</td><td>0</td></tr>
<tr><td>Estimator (추정기)</td><td><code>fit</code> · <code>predict</code>가 있는 모델</td><td>1-1</td></tr>
<tr><td>Transformer (변환기)</td><td><code>fit</code> · <code>transform</code>으로 데이터를 바꾸는 전처리기</td><td>머신러닝-2</td></tr>
<tr><td>Pipeline</td><td>전처리 + 모델을 묶어 "train으로만 fit"을 자동으로 지킴</td><td>머신러닝-2</td></tr>
<tr><td>Metric (평가지표)</td><td>성능을 숫자로 (분류 = 정확도 · F1, 회귀 = MAE · RMSE)</td><td>1-1 · 머신러닝-3</td></tr>
<tr><td>Hyperparameter (설정값)</td><td>사람이 정하는 값 (<code>max_depth</code> · <code>n_neighbors</code>)</td><td>4장</td></tr>
</tbody></table>

### 1-1. `A` 모델 사용 3단계 · `fit()` · `predict()` · `score()` <sub>원본 01 intro · 02 셀 2</sub>

> 🎯 **머신러닝 모델은 어떤 순서로 쓰며, 모델을 바꾸면 사용법도 바뀌는가?**

💡 **쉽게 말하면**

- 모델은 **문제집을 푸는 학생**.
- `fit`은 문제와 정답으로 공부하기, `predict`는 정답 없이 답 쓰기, `score`는 채점.

**실행 흐름**

1. 모델 만들기
2. fit(X, y)
3. predict(X)
4. score / accuracy_score
5. 다른 모델 2개도 같은 3줄

**먼저 예측** · `clf.score(X, y)`와 `accuracy_score(y, clf.predict(X))`는 같은 값일까? — 답은 코드 실행 결과 아래에

In [2]:
# 1단계: 모델 만들기 (설정값만 정함, 아직 학습 전)
# 왜 · 설정값을 정하는 단계와 학습 단계를 구분해 이해하기 위해
clf = DecisionTreeClassifier(random_state=SEED)  # seed 고정 → 같은 트리


# 2단계: 학습 → fit(문제, 정답)
# 왜 · 모델이 데이터에서 규칙을 배우는 단계를 확인하기 위해
clf.fit(X_full, y_full)  # 규칙을 모델 안에 저장


# 3단계: 예측 → predict(문제) → 정답 없이 답만 냄
# 왜 · 학습한 모델은 정답 없이 문제만으로 답을 낸다는 것을 보기 위해
pred = clf.predict(X_full[:5])               # 앞 5송이
print("예측:", pred, "| 정답:", y_full[:5])  # 예측과 정답 나란히


# 4단계: 채점 → 두 방법은 같은 값
# 왜 · 직접 계산한 정확도와 score()가 같은지 확인하기 위해
pred_all = clf.predict(X_full)                              # 150송이 모두 예측
print("accuracy_score:", accuracy_score(y_full, pred_all))  # (정답, 예측)
print("clf.score     :", clf.score(X_full, y_full))         # predict → 채점


# 모델을 바꿔도 fit → score 사용법은 같음
# 왜 · sklearn 모델은 사용법이 같아 모델만 바꿔 끼울 수 있음을 보이기 위해
others = [LogisticRegression(max_iter=2000),                       # 다른 모델 2개 준비
          KNeighborsClassifier(n_neighbors=5)]
for model in others:                                               # 모델마다 같은 사용법
    model.fit(X_full, y_full)                                      # 학습
    name = type(model).__name__                                    # 모델 클래스 이름
    print(f"{name:<22} score: {model.score(X_full, y_full):.3f}")  # 채점 (모델 이름 22칸)

예측: [0 0 0 0 0] | 정답: [0 0 0 0 0]
accuracy_score: 1.0
clf.score     : 1.0
LogisticRegression     score: 0.973
KNeighborsClassifier   score: 0.967


**결과** · 예측 정답은 같음 — `score`가 안에서 `predict` 후 `accuracy_score`를 계산

- `예측: [0 0 0 0 0] | 정답: [0 0 0 0 0]` — 앞 5송이 모두 맞힘
- `accuracy_score: 1.0` — 학습한 붓꽃 전부를 그대로 채점 → 만점 (1-2에서 이 숫자를 의심)
- `clf.score     : 1.0` — 같은 계산이라 같은 값
- `LogisticRegression     score: 0.973` — 모델을 바꿔도 `fit` · `score` 두 줄 그대로
- `KNeighborsClassifier   score: 0.967` — 모델마다 점수만 다름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>model.fit(X, y)</code></td><td>문제 + 정답으로 <strong>규칙을 학습</strong> (모델 안에 저장)</td><td>학습 데이터로 한 번</td></tr>
<tr><td><code>model.predict(X)</code></td><td>정답 없이 <strong>답만</strong> 냄</td><td>새 데이터 예측 · 틀린 샘플 찾기</td></tr>
<tr><td><code>model.score(X, y)</code></td><td>predict + 채점을 <strong>한 번에</strong> (분류 = 정확도)</td><td>점수만 빠르게 볼 때</td></tr>
<tr><td><code>accuracy_score(y, pred)</code></td><td>이미 만든 예측으로 <strong>채점만</strong></td><td>같은 예측으로 지표 여러 개를 낼 때 (3권 평가지표)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `accuracy_score(정답, 예측)` 순서
> - 정확도는 바꿔도 같지만 3권의 정밀도 · 재현율은 순서가 바뀌면 값이 달라짐

> ✅ **한 줄 정리** · 모든 모델은 만들기 → `fit(X, y)` → `predict(X)` → `score`의 같은 순서로 쓴다.

**딥러닝 코드에서는**

- PyTorch는 `fit`이 없고 직접 반복
- `for x, y in loader: loss = loss_fn(model(x), y)` → `loss.backward()` → `optimizer.step()`이 `fit` 한 번, `model(x)`가 `predict`

### 1-2. `A` 학습 데이터로 채점하는 착시 · `score(X_train)` <sub>원본 02 Part 1 · 빈칸 3~5</sub>

> 🎯 **학습한 데이터로 다시 채점하면 왜 실력을 알 수 없는가?**

💡 **쉽게 말하면**

- 공부한 문제집으로 그대로 시험을 보면 **답을 외운 학생도 100점**.
- 실력은 처음 보는 문제로만 잴 수 있음.

<img src="images/ml1_illusion.png" alt="학습한 데이터로 채점하면 엉터리 정답도 0.993" width="587">

**실행 흐름**

1. 트리 · KNN(k=1)을 전체로 학습
2. 같은 데이터로 채점
3. 정답을 무작위로 섞어 다시

**먼저 예측** · 꽃과 정답의 짝을 무작위로 섞어(배울 규칙이 없음) 학습한 트리는 학습 데이터에서 몇 점일까? — 답은 코드 실행 결과 아래에

In [3]:
# 원본 셀: 전체로 학습 → 같은 데이터로 채점
# 왜 · 학습에 쓴 데이터로 채점하면 점수가 부풀려지는 문제를 먼저 보이기 위해
clf = DecisionTreeClassifier(random_state=SEED).fit(X_full, y_full)     # fit은 모델 자신을 돌려줌
print(f"트리 · 학습한 데이터로 채점: {clf.score(X_full, y_full):.3f}")  # 외운 문제로 시험


# 이웃 1개만 보는 KNN도 같은 착시 (가장 가까운 이웃 = 자기 자신)
# 왜 · 모델 종류와 상관없이 같은 데이터 채점은 착시라는 것을 확인하기 위해
knn1 = KNeighborsClassifier(n_neighbors=1).fit(X_full, y_full)               # 이웃 1개만 보는 KNN
print(f"KNN(k=1) · 학습한 데이터로 채점: {knn1.score(X_full, y_full):.3f}")  # 자기 자신이 이웃


# 정답을 무작위로 섞어도 거의 100% → 규칙이 아니라 외운 것
# 왜 · 배울 규칙이 없어도 점수가 높게 나오면 외운 것이라는 증거를 보이기 위해
rng = np.random.default_rng(SEED)     # 섞는 순서 고정
y_shuffled = rng.permutation(y_full)  # 꽃 ↔ 정답 짝을 섞음 → 배울 규칙 없음
fake = DecisionTreeClassifier(random_state=SEED).fit(X_full, y_shuffled)  # 엉터리 정답으로 학습
# 규칙 없어도 높음?
print(f"엉터리 정답 · 학습한 데이터로 채점: {fake.score(X_full, y_shuffled):.3f}")

트리 · 학습한 데이터로 채점: 1.000
KNN(k=1) · 학습한 데이터로 채점: 1.000
엉터리 정답 · 학습한 데이터로 채점: 0.993


**결과** · 예측 정답은 `0.993` — 규칙이 없어도 거의 100%, 즉 이 점수는 **외운 양**이지 실력이 아님

- `트리 · 학습한 데이터로 채점: 1.000` — 끝까지 자란 트리는 학습 데이터를 전부 외움
- `KNN(k=1) · 학습한 데이터로 채점: 1.000` — 가장 가까운 이웃이 자기 자신 → 항상 정답
- `엉터리 정답 · 학습한 데이터로 채점: 0.993` — 1송이만 틀림 — 측정값이 똑같은 두 송이에 엉터리 정답이 서로 다르게 붙어 둘 다 맞힐 수는 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>학습한 데이터로 채점</td><td><strong>외운 문제</strong>로 시험 → 거의 항상 100%</td><td>쓰지 않음 (과적합 정도를 볼 때만)</td></tr>
<tr><td>처음 보는 데이터로 채점</td><td>외우지 못한 문제 → <strong>진짜 실력</strong></td><td>모든 평가 (2장부터)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 점수가 100%면 축하가 아니라 의심부터
> - 학습 데이터로 낸 점수는 보고하지 않음

> ✅ **한 줄 정리** · 학습한 데이터로 낸 점수는 외운 양일 뿐이라, 실력은 처음 보는 데이터로만 잰다.

**딥러닝 코드에서는**

- 학습 중 출력되는 `train_loss` · `train_acc`가 이 점수
- 그래서 매 epoch마다 `val_loss` · `val_acc`를 따로 계산함

---
## 2. 데이터 나누기

> **이 장의 흐름**
>
> - 1장에서 학습한 데이터로 낸 점수는 외운 양이었다
> - 처음 보는 데이터를 만들려면 test를 미리 떼어 봉인해야 함.
> - 그런데 한 번 떼어 낸 점수에도 운이 섞임

<img src="images/ml1_split_flow.png" alt="데이터 분할 흐름도" width="874">

**파이프라인 위치** · ② 나누기 — 학습용 · 최종 확인용 분리

### 2-1. `A` 8 : 2로 나누기 · `train_test_split()` <sub>원본 02 셀 9 · 빈칸 11~22</sub>

> 🎯 **train과 test는 어떻게 나누며, `stratify`와 `random_state`는 왜 넣는가?**

💡 **쉽게 말하면**

- `train_test_split`은 **문제집에서 시험지를 미리 뜯어 봉인**하는 것.
- `stratify`는 시험지에도 과목 비율을 똑같이, `random_state`는 같은 버전에서 매번 같은 장을 뜯기.

**실행 흐름**

1. 8 : 2 분할
2. stratify 있음 · 없음 test 클래스 비교
3. train으로 학습 · test로 채점
4. 1-2의 엉터리 모델도 test로

**먼저 예측** · `stratify` 없이 나누면 test 30송이의 클래스별 개수는 10 · 10 · 10일까? — 답은 코드 실행 결과 아래에

In [4]:
from sklearn.model_selection import train_test_split  # 나누는 함수


# 원본 셀: 8 : 2로 나누기 (stratify = 클래스 비율 유지, random_state = 고정)
# 왜 · 처음 보는 데이터로 채점하려고 test를 떼어 내고 클래스 비율을 유지하기 위해
X_train, X_test, y_train, y_test = train_test_split(     # 8 : 2로 나눔
    X_full, y_full, test_size=0.2, stratify=y_full,
    random_state=SEED)                                   # 반환 순서: X 둘 → y 둘
print("train:", X_train.shape, "| test:", X_test.shape)  # 120 · 30 확인


# stratify 있음 vs 없음 → test의 클래스별 개수
# 왜 · stratify가 클래스 비율을 지켜 주는지 개수로 확인하기 위해
_, _, _, y_test_plain = train_test_split(                        # 비교용 · y_test만 받음
    X_full, y_full, test_size=0.2, random_state=SEED)            # stratify 없이
print("stratify O · test 클래스별:", np.bincount(y_test))        # 클래스 비율 유지?
print("stratify X · test 클래스별:", np.bincount(y_test_plain))  # 무작위면 치우침


# 학습은 train, 채점은 test
# 왜 · 학습과 채점 데이터를 나눈 뒤의 점수를 착시 점수와 비교하기 위해
clf = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)  # 학습은 train으로만
# 외운 문제 · 처음 보는 문제
tr_acc, te_acc = clf.score(X_train, y_train), clf.score(X_test, y_test)
print(f"train 점수: {tr_acc:.3f} | test 점수: {te_acc:.3f}")           # 두 점수 비교


# 1-2의 엉터리 정답 모델도 test로 채점하면 정체가 드러남
# 왜 · 외운 모델은 처음 보는 데이터에서 무너진다는 것을 확인하기 위해
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(                       # 엉터리 정답도 같은 방법으로
    X_full, y_shuffled, test_size=0.2, random_state=SEED)
fake = DecisionTreeClassifier(random_state=SEED).fit(Xf_tr, yf_tr)   # train만 학습
tr_acc, te_acc = fake.score(Xf_tr, yf_tr), fake.score(Xf_te, yf_te)  # 엉터리 모델의 두 점수
print(f"엉터리 정답 · train 점수: {tr_acc:.3f} | test 점수: {te_acc:.3f}")  # test에서 정체가 드러남

train: (120, 4) | test: (30, 4)
stratify O · test 클래스별: [10 10 10]
stratify X · test 클래스별: [12  8 10]
train 점수: 1.000 | test 점수: 0.933
엉터리 정답 · train 점수: 0.992 | test 점수: 0.233


**결과** · 예측 정답은 아님 — `[12  8 10]`처럼 우연히 치우침

- `train: (120, 4) | test: (30, 4)` — 150 × 0.2 = 30송이를 test로 봉인
- `stratify O · test 클래스별: [10 10 10]` — 원본 비율 50 : 50 : 50 그대로
- `stratify X · test 클래스별: [12  8 10]` — versicolor가 8송이로 치우침
- `train 점수: 1.000 | test 점수: 0.933` — 외운 문제 만점 vs 처음 보는 문제 93.3% (30송이 중 2송이 틀림)
- `엉터리 정답 · train 점수: 0.992 | test 점수: 0.233` — test에서는 찍기(0.333)보다도 낮음 → 외운 것이 드러남

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>stratify=y</code></td><td>train · test의 <strong>클래스 비율을 원본과 같게</strong></td><td>분류 문제 (기본 선택)</td></tr>
<tr><td><code>stratify</code> 없음</td><td>무작위 → 비율이 <strong>우연히 치우칠 수 있음</strong></td><td>회귀 문제 (정답이 연속 숫자)</td></tr>
<tr><td><code>random_state=숫자</code></td><td>실행할 때마다 <strong>같은 분할</strong></td><td>결과를 재현 · 비교할 때 (항상)</td></tr>
<tr><td><code>test_size=0.2</code></td><td>전체의 20%를 test로</td><td>보통 0.2 ~ 0.3, 데이터가 적으면 교차검증과 함께</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 반환 순서는 `X_train, X_test, y_train, y_test` (X 둘 → y 둘), 순서를 틀려도 에러 없이 엉뚱한 결과

> ✅ **한 줄 정리** · `train_test_split(X, y, test_size=0.2, stratify=y, random_state=고정)`으로 나누고 test는 마지막까지 봉인한다.

**딥러닝 코드에서는**

- 이미지 폴더를 나눌 때도 같은 함수 · `train_paths, test_paths = train_test_split(paths, stratify=labels, random_state=42)`

### 2-2. `A` 한 번 나누기의 운 · `random_state` <sub>원본 02 Part 2 질문 · 빈칸 11~22</sub>

> 🎯 **train · test를 한 번만 나누면 그 점수를 믿어도 되는가?**

💡 **쉽게 말하면**

- 시험 **한 번**은 그날 문제 운이 섞임.
- 같은 학생(모델)인데 시험지(분할)만 바꿔도 점수가 오르내림.

**실행 흐름**

1. 분할 seed 0 ~ 9로 10번 나누기
2. 같은 모델로 test 점수 10개
3. 최저 · 최고 비교

**먼저 예측** · 모델은 그대로 두고 나누는 seed만 바꾸면 test 점수는 같을까? — 답은 코드 실행 결과 아래에

In [5]:
# 원본 Part 2 질문: 한 번만 나누면 충분할까? → 나누는 seed만 바꿔 보기
# 왜 · 분할 seed만 바꿔도 test 점수가 흔들린다는 것을 확인하기 위해
split_scores = []                               # seed별 test 점수를 모을 곳
for rs in range(10):                            # 분할 seed 0 ~ 9, 모델 설정은 그대로
    Xtr, Xte, ytr, yte = train_test_split(      # 시험지만 바꿔 나누기
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=rs)
    model = DecisionTreeClassifier(random_state=SEED).fit(Xtr, ytr)  # 모델 설정은 그대로
    split_scores.append(model.score(Xte, yte))  # test 점수 기록


# 같은 모델인데 시험지만 바뀌어도 점수가 흔들림
# 왜 · 한 번의 분할로 결론 내리면 위험하다는 것을 숫자로 보기 위해
print("seed별 test 점수:", np.round(split_scores, 3))         # 10개 점수
lo, hi = min(split_scores), max(split_scores)                 # 가장 낮은 · 높은 점수
print(f"최저 {lo:.3f} · 최고 {hi:.3f} · 차이 {hi - lo:.3f}")  # 흔들림 크기
print(f"test 30개 중 1개 = {1 / 30:.3f}")                     # 점수 한 칸의 크기

seed별 test 점수: [0.967 0.967 0.933 0.9   0.967 0.967 0.9   0.933 0.967 0.933]
최저 0.900 · 최고 0.967 · 차이 0.067
test 30개 중 1개 = 0.033


**결과** · 예측 정답은 다름 — `0.900` ~ `0.967`로 흔들림

- `seed별 test 점수: [0.967 0.967 0.933 0.9   0.967 0.967 0.9   0.933 0.967 0.933]` — 같은 모델인데 시험지만 바뀜
- `최저 0.900 · 최고 0.967 · 차이 0.067` — 30송이 중 2송이 차이 = 실력이 아니라 운
- `test 30개 중 1개 = 0.033` — test가 작으면 1개만 틀려도 3.3%p가 움직임

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>test 점수 1개</td><td>분할 <strong>운</strong>이 섞인 값</td><td>최종 보고 1회 (5장)</td></tr>
<tr><td>여러 분할의 평균 · 흔들림</td><td>운을 <strong>평균으로 줄인</strong> 값</td><td>모델 · 설정을 고를 때 (3장 교차검증)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - test 점수 하나로 "A 모델이 B보다 3%p 좋다"고 결론 내리기
> - 분할 운일 수 있음

> ✅ **한 줄 정리** · 한 번 나눈 test 점수에는 분할 운이 섞이므로, 모델을 고를 때는 여러 번 나눠 평균을 본다.

**딥러닝 코드에서는**

- 딥러닝도 seed에 따라 val 점수가 흔들림
- 논문 · 대회에서 seed 3~5개 평균을 보고하는 이유

---
## 3. 교차검증

> **이 장의 흐름**
>
> - 2장에서 한 번 나눈 점수는 분할 운에 따라 흔들렸다
> - train 안에서 여러 번 나눠 평균을 내면 운이 줄어듦.
> - 나누는 방법을 고르고, 한 줄 함수로 줄이고, 모델끼리 비교

**파이프라인 위치** · ② 나누기 · ⑥ 평가 · 선택 — 같은 분할로 공정하게 비교

### 3-1. `A` K-Fold 직접 돌리기 · `StratifiedKFold.split()` <sub>원본 02 셀 26</sub>

> 🎯 **교차검증은 데이터를 어떻게 나누고 몇 번 학습하는가?**

💡 **쉽게 말하면**

- **모의고사 5번**.
- train을 5등분해 매번 1등분을 모의고사지로, 나머지 4등분을 교과서로 써서 5번 보고 평균을 냄.

<img src="images/ml1_kfold.png" alt="5-Fold · 매번 다른 1등분이 검증" width="689">

**실행 흐름**

1. StratifiedKFold(5)
2. split으로 번호 5쌍
3. fold마다 학습 · 검증
4. 평균 ± 표준편차
5. 모든 번호가 한 번씩 검증인지 확인

**먼저 예측** · train 120개를 5-Fold로 나누면 fold마다 학습 · 검증은 몇 개씩일까? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.model_selection import StratifiedKFold  # 클래스 비율 유지 K등분


# 5등분 규칙 만들기 (클래스 비율 유지 · 섞기 · seed 고정)
# 왜 · 교차검증의 등분 방식을 한 번 정해 모든 비교에 같게 쓰기 위해
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # 규칙만 만듦 (아직 안 나눔)


# 5번 반복: 4등분으로 학습 → 남은 1등분으로 검증 (test는 그대로 봉인)
# 왜 · 교차검증이 하는 일을 반복문으로 직접 풀어 이해하기 위해
fold_scores = []                                             # fold별 점수를 모을 곳
folds = skf.split(X_train, y_train)                          # (학습 번호, 검증 번호) 5쌍
for fold, (tr_idx, val_idx) in enumerate(folds, 1):          # 1부터 번호
    model = DecisionTreeClassifier(random_state=SEED)        # fold마다 새 모델
    model.fit(X_train[tr_idx], y_train[tr_idx])              # 4등분으로 학습
    score = model.score(X_train[val_idx], y_train[val_idx])  # 남은 1등분으로 채점
    fold_scores.append(score)                                # 점수 기록
    print(f"Fold {fold}: 학습 {len(tr_idx)}개 · 검증 {len(val_idx)}개"  # fold별 개수 · 점수
          f" → {score:.3f}")
print(f"평균 {np.mean(fold_scores):.3f}"                     # 실력 = 평균 · 흔들림 = 표준편차
      f" ± 표준편차 {np.std(fold_scores):.3f}")


# 모든 샘플이 정확히 한 번씩 검증에 쓰였는지 확인
# 왜 · 등분이 빠짐없이 · 겹침 없이 됐는지 검증하기 위해
all_val = np.concatenate([v for _, v in skf.split(X_train, y_train)])  # 검증 번호를 모두 이어 붙임
print("검증에 쓰인 번호:", len(all_val), "개 · 중복 없음:",            # 모두 한 번씩인가
      len(set(all_val)) == len(X_train))

Fold 1: 학습 96개 · 검증 24개 → 0.917
Fold 2: 학습 96개 · 검증 24개 → 0.917
Fold 3: 학습 96개 · 검증 24개 → 1.000
Fold 4: 학습 96개 · 검증 24개 → 1.000
Fold 5: 학습 96개 · 검증 24개 → 0.958
평균 0.958 ± 표준편차 0.037
검증에 쓰인 번호: 120 개 · 중복 없음: True


**결과** · 예측 정답은 학습 `96`개 · 검증 `24`개 (120 ÷ 5 = 24)

- `Fold 1: 학습 96개 · 검증 24개 → 0.917` — train 120개를 4 : 1로
- `Fold 3: 학습 96개 · 검증 24개 → 1.000` — fold마다 0.917 ~ 1.000으로 다름
- `평균 0.958 ± 표준편차 0.037` — 점수 5개의 평균 실력 · 흔들림
- `검증에 쓰인 번호: 120 개 · 중복 없음: True` — 모든 샘플이 정확히 한 번씩 모의고사지가 됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>skf.split(X, y)</code></td><td>(학습 번호, 검증 번호)를 <strong>직접 받음</strong></td><td>fold마다 다른 처리가 필요할 때 (딥러닝 K-Fold 학습)</td></tr>
<tr><td><code>cross_val_score</code></td><td>같은 반복을 <strong>한 줄로</strong> (3-3)</td><td>점수만 필요할 때 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 교차검증은 **train 안에서만**
> - 원본 셀 26은 전체 X로 돌렸지만 정리본은 test를 봉인한 채 X_train으로

> ✅ **한 줄 정리** · train을 K등분해 1등분씩 돌아가며 검증하고, K개 점수의 평균 ± 표준편차로 실력을 본다.

**딥러닝 코드에서는**

- 딥러닝 K-Fold도 같은 뼈대
- `for fold, (tr, va) in enumerate(skf.split(X, y)):` 안에서 `Subset(dataset, tr)`으로 loader를 만들어 fold마다 새 모델 학습

### 3-2. `A` 나누는 방법 고르기 · `KFold` · `StratifiedKFold` <sub>원본 02 빈칸 29~36</sub>

> 🎯 **`KFold`와 `StratifiedKFold`는 무엇이 다르며, `shuffle`을 빼면 무슨 일이 생기는가?**

💡 **쉽게 말하면**

- 정답이 **0번 → 1번 → 2번 순서로 꽂힌 책장**을 앞에서부터 세 칸씩 자르면 칸마다 한 종류만 들어감.
- 섞거나(shuffle) 종류별로 고르게 나눠야(Stratified) 함.

**실행 흐름**

1. 정렬된 X_full을 3등분 3가지 방법
2. fold별 검증 클래스 개수 · 정확도 비교

**먼저 예측** · 정렬된 붓꽃을 섞지 않은 `KFold(3)`로 나누면 정확도는? — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.model_selection import KFold  # 클래스 비율을 안 보는 K등분


# 정답이 0 → 1 → 2로 정렬된 원래 순서(X_full) 그대로 3등분해 보기
# 왜 · 섞지 않은 KFold가 정렬된 데이터에서 0점 fold를 만드는 위험을 보이기 위해
cvs = {  # 비교할 나누는 방법 3가지
    "KFold · 섞지 않음": KFold(n_splits=3),
    "KFold · 섞음": KFold(n_splits=3, shuffle=True, random_state=SEED),
    "StratifiedKFold": StratifiedKFold(
        n_splits=3, shuffle=True, random_state=SEED),
}


# 방법마다 fold별 검증 클래스 개수 · 정확도
# 왜 · 섞기 · 층화 여부에 따라 fold 구성이 어떻게 달라지는지 비교하기 위해
for name, cv in cvs.items():                                        # 방법마다
    counts, accs = [], []                                           # fold별 클래스 개수 · 정확도
    for tr_idx, val_idx in cv.split(X_full, y_full):                # KFold는 y를 안 씀
        model = DecisionTreeClassifier(random_state=SEED)           # fold마다 새 모델
        model.fit(X_full[tr_idx], y_full[tr_idx])                   # 학습 fold로만
        accs.append(model.score(X_full[val_idx], y_full[val_idx]))  # 검증 fold 정확도
        cnt = np.bincount(y_full[val_idx], minlength=3)             # 0개 클래스도 표시
        counts.append(cnt.tolist())                                 # 출력용 리스트로
    print(f"{name} → 검증 클래스별 {counts} · 정확도 {np.round(accs, 3)}")  # 방법별 결과 한 줄

KFold · 섞지 않음 → 검증 클래스별 [[50, 0, 0], [0, 50, 0], [0, 0, 50]] · 정확도 [0. 0. 0.]
KFold · 섞음 → 검증 클래스별 [[16, 16, 18], [17, 14, 19], [17, 20, 13]] · 정확도 [0.98 0.98 0.92]
StratifiedKFold → 검증 클래스별 [[17, 17, 16], [17, 16, 17], [16, 17, 17]] · 정확도 [0.94 0.98 0.94]


**결과** · 예측 정답은 `[0. 0. 0.]` — 검증 fold의 클래스가 학습에 하나도 없음

- `KFold · 섞지 않음 → 검증 클래스별 [[50, 0, 0], [0, 50, 0], [0, 0, 50]] · 정확도 [0. 0. 0.]` — 검증이 학습에 없는 클래스 하나로만 채워져 전부 0점
- `KFold · 섞음 → 검증 클래스별 [[16, 16, 18], [17, 14, 19], [17, 20, 13]] · 정확도 [0.98 0.98 0.92]` — 섞으면 해결되지만 클래스 개수가 13 ~ 20으로 들쭉날쭉
- `StratifiedKFold → 검증 클래스별 [[17, 17, 16], [17, 16, 17], [16, 17, 17]] · 정확도 [0.94 0.98 0.94]` — fold마다 16 ~ 17로 고르게

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>train_test_split</code></td><td><strong>1번</strong> 나눔 (train · test)</td><td>최종 test 떼기 · 데이터가 충분히 클 때의 검증</td></tr>
<tr><td><code>KFold(shuffle=True)</code></td><td>K번 나눔, <strong>클래스 비율은 신경 쓰지 않음</strong></td><td>회귀 문제 · 클래스가 고르게 섞인 데이터</td></tr>
<tr><td><code>KFold()</code> (섞지 않음)</td><td>앞에서부터 순서대로 자름 → <strong>정렬된 데이터면 사고</strong></td><td>시간 순서를 지켜야 할 때만 (시계열은 <code>TimeSeriesSplit</code>)</td></tr>
<tr><td><code>StratifiedKFold</code></td><td>K번 나눔 + <strong>fold마다 클래스 비율 유지</strong></td><td>분류 문제 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - CSV는 클래스 · 날짜 순으로 정렬된 경우가 많음
> - `KFold()` 기본값(섞지 않음)을 그대로 쓰면 이 사고가 조용히 일어남

> ✅ **한 줄 정리** · 분류 문제는 `StratifiedKFold(shuffle=True, random_state=고정)`으로 나눠 fold마다 클래스 비율을 맞춘다.

**딥러닝 코드에서는**

- 폴더별로 정렬된 이미지 목록도 똑같음
- `DataLoader(shuffle=True)`로 학습 배치를 섞는 이유, 시계열만 예외로 시간 순서를 지킴

### 3-3. `A` 교차검증 한 줄 · `cross_val_score()` <sub>원본 02 셀 46 · 빈칸 38~43</sub>

> 🎯 **3-1의 for문을 한 줄로 쓰려면, 그리고 `cv=5`라고만 쓰면 어떻게 나뉘는가?**

💡 **쉽게 말하면**

- `cross_val_score`는 **모의고사 자동 채점기**.
- 모델 · 데이터 · 나누는 규칙(cv)을 넣으면 fold별 점수 배열을 돌려줌.

**실행 흐름**

1. 같은 skf로 cross_val_score
2. 3-1과 비교
3. cv=5와 StratifiedKFold(5) 비교

**먼저 예측** · `cv=5`(숫자)로 넣으면 3-1의 섞은 5등분과 같은 점수가 나올까? — 답은 코드 실행 결과 아래에

In [8]:
from sklearn.model_selection import cross_val_score  # 교차검증 한 줄 함수


# 3-1의 for문을 한 줄로 (같은 skf를 넘기면 결과도 같음)
# 왜 · 직접 만든 반복문과 cross_val_score가 같은 결과인지 확인하기 위해
tree = DecisionTreeClassifier(random_state=SEED)              # 학습 전 모델을 넘김 (안에서 복사)
scores = cross_val_score(tree, X_train, y_train, cv=skf,      # 3-1과 같은 5등분
                         scoring="accuracy")                  # 모델 · X · y · 분할 · 채점
print("각 Fold:", scores.round(3))                            # fold별 점수 배열
print("3-1 for문과 같음:", np.allclose(scores, fold_scores))  # 실수 비교


# cv=5 (숫자)는 섞지 않는 StratifiedKFold(5)와 같음 (분류 모델일 때)
# 왜 · cv에 숫자만 넣으면 섞지 않는다는 기본 동작을 확인하기 위해
s_int = cross_val_score(tree, X_train, y_train, cv=5)                   # 숫자만 넘김
s_obj = cross_val_score(tree, X_train, y_train, cv=StratifiedKFold(5))  # 섞지 않는 객체
print(f"cv=5 평균 {s_int.mean():.3f}"                                   # 두 방법이 같은지
      f" | StratifiedKFold(5) 평균 {s_obj.mean():.3f}"
      f" | 같음: {np.allclose(s_int, s_obj)}")

각 Fold: [0.917 0.917 1.    1.    0.958]
3-1 for문과 같음: True
cv=5 평균 0.942 | StratifiedKFold(5) 평균 0.942 | 같음: True


**결과** · 예측 정답은 아님 — 숫자 `5`는 **섞지 않는** `StratifiedKFold(5)`라서 평균이 `0.942`로 다름

- `각 Fold: [0.917 0.917 1.    1.    0.958]` — 3-1의 점수 5개와 같은 순서 · 같은 값
- `3-1 for문과 같음: True` — for문 8줄 = 한 줄
- `cv=5 평균 0.942 | StratifiedKFold(5) 평균 0.942 | 같음: True` — 섞지 않은 5등분이라 3-1의 0.958과 다름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>cv=5</code> (숫자)</td><td>분류면 <strong>섞지 않는</strong><code>StratifiedKFold(5)</code></td><td>데이터가 이미 섞여 있을 때</td></tr>
<tr><td><code>cv=StratifiedKFold(5, shuffle=True, random_state=…)</code></td><td>섞기 · seed까지 <strong>직접 지정</strong></td><td>모델끼리 <strong>같은 분할</strong>로 비교할 때 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 46은 주석이 5-Fold인데 코드는 `cv=4`, 저장된 출력은 7개
> - 정리본은 `cv=skf`(5등분) 하나로 통일

> ✅ **한 줄 정리** · `cross_val_score(모델, X, y, cv=분할기)`로 교차검증을 한 줄로 하고, 모델끼리 비교할 때는 같은 분할기를 넘긴다.

**딥러닝 코드에서는**

- PyTorch에는 이 함수가 없어 3-1의 for문을 직접 씀
- 그래서 원리(3-1)를 먼저 익힘

### 3-4. `B` 지표 여러 개 · 학습 점수 · `cross_validate()` <sub>원본 02 빈칸 38~43</sub>

> 🎯 **정확도 말고 다른 지표와 학습 점수까지 한 번에 보려면?**

💡 **쉽게 말하면**

- `cross_validate`는 **성적표 전체**.
- `cross_val_score`가 과목 하나의 점수라면, 여러 과목 · 공부한 문제 점수 · 걸린 시간까지 dict로 줌.

**실행 흐름**

1. cross_validate(지표 2개, return_train_score=True)
2. 키 확인
3. 학습 · 검증 평균
4. cross_val_score와 비교

**먼저 예측** · 결과 dict에서 `test_accuracy`는 봉인한 최종 test 점수일까? — 답은 코드 실행 결과 아래에

In [9]:
from sklearn.model_selection import cross_validate  # 지표 여러 개 교차검증


# 지표 여러 개 + 학습 점수까지 한 번에 → 결과 = dict
# 왜 · 학습 점수와 검증 점수를 함께 봐야 과적합을 판단할 수 있기 때문에
res = cross_validate(tree, X_train, y_train, cv=skf,  # 같은 5등분
                     scoring=["accuracy", "f1_macro"],
                     return_train_score=True)
print("돌려주는 키:", sorted(res))                    # dict의 키 이름 확인


# test_ = 각 fold의 검증 점수 (최종 test 아님), train_ = 학습한 4등분 점수
# 왜 · 결과 이름의 test_가 최종 test가 아니라는 혼동을 막기 위해
tr_mean = res["train_accuracy"].mean()                                       # 학습 4등분 점수 평균
val_mean = res["test_accuracy"].mean()                                       # 검증 1등분 점수 평균
print(f"학습 정확도 평균 {tr_mean:.3f} | 검증 정확도 평균 {val_mean:.3f}")   # 차이 = 외운 정도
print(f"검증 F1(macro) 평균 {res['test_f1_macro'].mean():.3f}")              # 두 번째 지표
print("cross_val_score와 같음:", np.allclose(res["test_accuracy"], scores))  # 3-3과 같은 값?

돌려주는 키: ['fit_time', 'score_time', 'test_accuracy', 'test_f1_macro', 'train_accuracy', 'train_f1_macro']
학습 정확도 평균 1.000 | 검증 정확도 평균 0.958
검증 F1(macro) 평균 0.958
cross_val_score와 같음: True


**결과** · 예측 정답은 아님 — 각 fold의 **검증** 점수 (이름만 test)

- `돌려주는 키: ['fit_time', 'score_time', 'test_accuracy', 'test_f1_macro', 'train_accuracy', 'train_f1_macro']` — 지표마다 `test_` · `train_` 두 개씩
- `학습 정확도 평균 1.000 | 검증 정확도 평균 0.958` — 차이 0.042 = 외운 정도 (4-1에서 활용)
- `검증 F1(macro) 평균 0.958` — 클래스가 고르면 정확도와 비슷 (F1은 머신러닝-3)
- `cross_val_score와 같음: True` — `test_accuracy` = 3-3의 점수 배열

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>cross_val_score</code></td><td>지표 <strong>1개</strong>의 fold별 점수 배열</td><td>점수 하나로 비교할 때 (기본 선택)</td></tr>
<tr><td><code>cross_validate</code></td><td>지표 <strong>여러 개</strong> + 학습 점수 + 시간을 dict로</td><td>과적합 확인(train vs 검증) · 정확도와 F1을 함께 볼 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `test_` 접두어는 fold의 검증 점수
> - 최종 test(X_test)와 헷갈리지 않기

> ✅ **한 줄 정리** · 지표가 여러 개거나 학습 점수가 필요하면 `cross_validate`, 점수 하나면 `cross_val_score`를 쓴다.

**딥러닝 코드에서는**

- 학습 기록 `history = {"train_loss": [...], "val_loss": [...], "val_f1": [...]}`와 같은 구조
- 두 곡선의 간격으로 과적합 판단

### 3-5. `A` 모델 비교와 선택 · `CV mean · std` <sub>원본 02 셀 47 · 49 · 53 · 54 · 빈칸 50 · 51</sub>

> 🎯 **교차검증 점수로 모델을 고를 때 평균 말고 무엇을 더 봐야 하는가?**

💡 **쉽게 말하면**

- 평균은 **실력**, 표준편차는 **기복**.
- 실력이 비슷하면 시험마다 점수가 덜 출렁이는 학생을 뽑음.

**실행 흐름**

1. 같은 skf로 모델 3개 mean · std 표
2. 1 · 2위 차이 1%p 규칙
3. 분할 seed 0 · 1 · 2로 다시 비교
4. 트리 seed만 바꿔 보기

**먼저 예측** · seed 18 분할에서 1위인 KNN은 분할 seed를 바꿔도 계속 1위일까? — 답은 코드 실행 결과 아래에

In [10]:
# 비교할 모델 3개 (원본과 같은 설정)
# 왜 · 원본과 같은 설정으로 세 모델을 같은 조건에서 비교하기 위해
models = {  # 비교할 모델 3개
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}


# 같은 5등분(skf)으로 비교 → mean 높은 순, 같으면 std 작은 순
# 왜 · 모든 모델이 같은 등분을 써야 점수 차이가 모델 차이라고 말할 수 있기 때문에
rows = []                                                                # 모델별 결과를 모을 곳
for name, model in models.items():                                       # 모델마다
    s = cross_val_score(model, X_train, y_train, cv=skf)                 # 같은 5등분으로 공정하게
    rows.append({"모델": name, "CV mean": s.mean(), "CV std": s.std()})  # 평균 · 흔들림 기록
df_cmp = pd.DataFrame(rows).sort_values(                                 # 표로 바꿔 정렬
    ["CV mean", "CV std"], ascending=[False, True])
print(df_cmp.round(3).to_string(index=False))                            # 비교표 출력


# 선택 규칙: 1 · 2위 mean 차이가 1%p 이하면 std가 작은 쪽
# 왜 · 점수를 본 뒤 마음대로 고르지 않도록 선택 규칙을 미리 정하기 위해
top1, top2 = df_cmp.iloc[0], df_cmp.iloc[1]                       # 1 · 2위 꺼내기
gap = (top1["CV mean"] - top2["CV mean"]) * 100                   # 0.01 → 1%p
if gap <= 1.0:                                                    # 차이가 작으면
    chosen = top1 if top1["CV std"] <= top2["CV std"] else top2   # 흔들림이 작은 쪽
else:
    chosen = top1                                                 # 차이가 크면 평균 1위
print(f"1 · 2위 mean 차이 {gap:.1f}%p → 선택: {chosen['모델']}")  # 규칙 적용 결과


# 5등분 seed를 바꿔도 같은 결론인지 확인
# 왜 · 결론이 한 가지 등분에만 기대고 있지 않은지 확인하기 위해
for cv_seed in [0, 1, 2]:                                          # 나누는 seed만 바꿔 보기
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)  # 다른 5등분
    means = {}                                                     # seed별 평균을 모을 곳
    for name, m in models.items():                                 # 모델마다
        mean = cross_val_score(m, X_train, y_train, cv=cv).mean()  # 그 5등분의 평균
        means[name] = float(mean.round(3))                         # np.float64 → float (출력용)
    print(f"cv seed {cv_seed}:", means)                            # 순위가 유지되나


# 원본 셀 49: 분할은 그대로 두고 트리 모델의 seed만 바꾸기
# 왜 · 모델 seed만으로도 점수가 흔들린다는 것을 분할 효과와 나눠 보기 위해
tree_means = []                                 # 트리 seed별 평균
for s in range(5):                              # 트리 seed 0 ~ 4
    m = DecisionTreeClassifier(random_state=s)  # 같은 점수일 때 고르는 순서
    tree_means.append(float(cross_val_score(m, X_train, y_train, cv=skf)  # 분할은 고정
                            .mean().round(3)))
print("트리 seed 0 ~ 4:", tree_means)           # 모델 seed의 흔들림

                모델  CV mean  CV std
         KNN (k=5)    0.983   0.033
LogisticRegression    0.967   0.041
      DecisionTree    0.958   0.037
1 · 2위 mean 차이 1.7%p → 선택: KNN (k=5)


cv seed 0: {'DecisionTree': 0.967, 'LogisticRegression': 0.967, 'KNN (k=5)': 0.958}


cv seed 1: {'DecisionTree': 0.95, 'LogisticRegression': 0.967, 'KNN (k=5)': 0.958}


cv seed 2: {'DecisionTree': 0.958, 'LogisticRegression': 0.975, 'KNN (k=5)': 0.967}
트리 seed 0 ~ 4: [0.967, 0.967, 0.967, 0.958, 0.958]


**결과** · 예측 정답은 아님 — seed 0 · 1에서는 KNN이 3위, seed 2에서는 로지스틱이 1위

- `KNN (k=5)    0.983   0.033` — seed 18 분할에서는 mean 1위 · std 최소
- `1 · 2위 mean 차이 1.7%p → 선택: KNN (k=5)` — 1%p보다 커서 mean이 높은 쪽
- `cv seed 0: {'DecisionTree': 0.967, 'LogisticRegression': 0.967, 'KNN (k=5)': 0.958}` — seed만 바꿨는데 KNN이 꼴찌
- `cv seed 2: {'DecisionTree': 0.958, 'LogisticRegression': 0.975, 'KNN (k=5)': 0.967}` — 이번엔 로지스틱이 1위 → 세 모델은 사실상 비슷한 실력
- `트리 seed 0 ~ 4: [0.967, 0.967, 0.967, 0.958, 0.958]` — 분할이 같아도 모델 seed만으로 0.009 차이 (원본 셀 49와 같은 결론)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>CV mean</td><td>평균 실력</td><td>1순위 기준</td></tr>
<tr><td>CV std</td><td>분할에 따른 <strong>흔들림</strong></td><td>mean 차이가 1~2%p 이내면 std 작은 쪽</td></tr>
<tr><td>분할 seed를 바꿔 재확인</td><td>순위가 <strong>우연인지</strong> 확인</td><td>후보끼리 점수 차이가 작을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 점수 차이가 test 몇 개 차이(여기서는 검증 1개 = 0.008)보다 작으면 우열이 아님

> ✅ **한 줄 정리** · 모델은 같은 분할의 CV mean으로 비교하되, 차이가 작으면 std와 다른 seed 결과까지 보고 고른다.

**딥러닝 코드에서는**

- 딥러닝 모델 비교도 같은 val 세트 · 같은 seed에서, 차이가 작으면 seed를 바꿔 반복해 확인

---
## 4. 하이퍼파라미터 탐색

> **이 장의 흐름**
>
> - 3장에서 교차검증 점수로 모델을 공정하게 비교했다
> - 같은 모델 안의 설정값(하이퍼파라미터)도 같은 방법으로 비교.
> - 손으로 반복하던 것을 자동 실험기로

**파이프라인 위치** · ④ 모델 · ⑥ 평가 · 선택 — 설정값 고르기, 과적합 확인

### 4-1. `A` 설정값과 과적합 · `max_depth` <sub>원본 02 셀 56 · 57</sub>

> 🎯 **트리의 깊이(`max_depth`)를 바꾸면 학습 점수와 검증 점수는 어떻게 움직이는가?**

💡 **쉽게 말하면**

- 깊이는 **질문 개수 한도**.
- 너무 적으면 기본도 못 배우고(과소적합), 너무 많으면 문제집 오타까지 외움(과적합).

<img src="images/ml1_depth.png" alt="깊이별 학습 점수와 검증 점수" width="587">

**실행 흐름**

1. 깊이 1 · 2 · 3 · 4 · None마다 cross_validate
2. 학습 · 검증 · 차이 표
3. 검증 최고 깊이

**먼저 예측** · 깊이를 끝까지(None) 늘리면 검증 점수도 계속 오를까? — 답은 코드 실행 결과 아래에

In [11]:
# 원본 셀: max_depth 후보마다 교차검증 (None = 끝까지 자람)
# 왜 · 깊이에 따라 검증 점수가 어떻게 바뀌는지 같은 CV로 비교하기 위해
rows = []                                                               # 깊이별 결과를 모을 곳
for d in [1, 2, 3, 4, None]:                                            # 깊이 후보마다
    model = DecisionTreeClassifier(max_depth=d, random_state=SEED)      # 깊이만 다른 트리
    r = cross_validate(model, X_train, y_train, cv=skf,                 # 학습 점수도 함께
                       return_train_score=True)
    rows.append({"max_depth": str(d), "학습": r["train_score"].mean(),  # 두 점수 기록
                 "검증": r["test_score"].mean()})


# 학습 점수와 검증 점수의 차이 = 외운 정도
# 왜 · 두 점수의 차이로 과적합 정도를 숫자로 읽기 위해
df_depth = pd.DataFrame(rows)                             # 표로 바꿈
df_depth["차이"] = df_depth["학습"] - df_depth["검증"]    # 외운 정도
print(df_depth.round(3).to_string(index=False))           # 깊이별 표
best_row = df_depth["검증"].idxmax()                      # 최댓값이 여러 개면 처음 나온 행
print("검증 최고:", df_depth.loc[best_row, "max_depth"])  # 고를 깊이

max_depth    학습    검증    차이
        1 0.667 0.667 0.000
        2 0.975 0.958 0.017
        3 0.985 0.958 0.027
        4 1.000 0.958 0.042
     None 1.000 0.958 0.042
검증 최고: 2


**결과** · 예측 정답은 아님 — 검증은 깊이 2부터 `0.958`에서 멈추고 학습 점수만 `1.000`까지 올라 차이가 커짐

- `1 0.667 0.667 0.000` — 질문 1개 → 둘 다 낮음 (과소적합)
- `2 0.975 0.958 0.017` — 검증 최고 · 차이 작음
- `4 1.000 0.958 0.042` — 학습 만점, 검증은 그대로 → 차이만 커짐 (외우기 시작)
- `검증 최고: 2` — 동점이면 먼저 나온 것 = 가장 단순한 깊이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>과소적합</td><td>학습 · 검증 <strong>둘 다 낮음</strong> (너무 단순)</td><td>깊이 · 복잡도를 늘림</td></tr>
<tr><td>적당함</td><td>검증이 <strong>가장 높고</strong> 학습과 차이가 작음</td><td>이 설정을 고름</td></tr>
<tr><td>과적합</td><td>학습은 높은데 <strong>검증과 차이가 큼</strong> (외움)</td><td>깊이를 줄이거나 규제 · 데이터 추가</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 검증 점수가 같으면 더 단순한 설정(얕은 트리)
> - 복잡한 쪽은 새 데이터에서 무너질 위험이 큼

> ✅ **한 줄 정리** · 학습 · 검증이 둘 다 낮으면 과소적합, 학습만 높고 차이가 크면 과적합이며, 검증이 가장 높은 설정을 고른다.

**딥러닝 코드에서는**

- 학습 곡선에서 `train_loss`는 계속 내려가는데 `val_loss`가 다시 오르기 시작하는 epoch = 과적합 시작 → 조기 종료(early stopping)

### 4-2. `A` 후보 자동 탐색 · `GridSearchCV` <sub>원본 02 셀 59 ~ 62</sub>

> 🎯 **하이퍼파라미터 후보를 교차검증으로 한 번에 비교하고 최적 모델까지 얻으려면?**

💡 **쉽게 말하면**

- `GridSearchCV`는 **자동 실험기**.
- 내가 적어 준 후보표의 칸마다 모의고사 5번을 보고, 1등 설정으로 train 전체를 다시 공부시켜 둠.

**실행 흐름**

1. 후보 4개 × 5등분 = 20번 학습
2. best_params_ · best_score_
3. cv_results_ 표 (학습 · 검증 · 순위)

**먼저 예측** · `GridSearchCV`가 고른 깊이와 CV 점수는 4-1 for문 결과와 같을까? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.model_selection import GridSearchCV  # 모든 조합 탐색기


# 탐색할 후보 (원본과 같은 4개)
# 왜 · 원본과 같은 후보로 결과를 비교하기 위해
param_grid = {"max_depth": [2, 3, 4, None]}  # 설정 이름: 후보 목록


# 후보 4개 × 5등분 = 20번 학습 + 최적 설정으로 X_train 전체에 1번 더
# 왜 · 직접 한 탐색을 GridSearchCV로 자동화하고 최적 설정으로 다시 학습하기 위해
grid = GridSearchCV(                    # 모델 · 후보 · 분할 · 채점 기준
    DecisionTreeClassifier(random_state=SEED), param_grid, cv=skf,
    scoring="accuracy",
    return_train_score=True)            # 원본은 주석 → 아래 표에서 KeyError
grid.fit(X_train, y_train)              # 20번 학습 + refit 1번
print("최적 설정:", grid.best_params_,  # 고른 설정 · CV 점수
      "| 최적 CV 점수:", round(grid.best_score_, 3))


# 모든 후보의 결과 표 (원본 셀 그대로, 순위순)
# 왜 · 1등만이 아니라 모든 후보의 점수 · 흔들림을 함께 보기 위해
cols = {"param_max_depth": "max_depth", "mean_train_score": "학습",      # 보여 줄 열 · 한글 이름
        "mean_test_score": "검증", "std_test_score": "표준편차",
        "rank_test_score": "순위"}
cv_df = pd.DataFrame(grid.cv_results_)[list(cols)].rename(columns=cols)  # 전체 결과 → 표
print(cv_df.sort_values("순위").round(3).to_string(index=False))         # 순위순 출력

최적 설정: {'max_depth': 2} | 최적 CV 점수: 0.958
max_depth    학습    검증  표준편차  순위
        2 0.975 0.958 0.037   1
        3 0.985 0.958 0.037   2
        4 1.000 0.958 0.037   2
     None 1.000 0.958 0.037   2


**결과** · 예측 정답은 같음 — 깊이 `2` · `0.958` (같은 skf를 썼으므로)

- `최적 설정: {'max_depth': 2} | 최적 CV 점수: 0.958` — 4-1 for문과 같은 결론
- `2 0.975 0.958 0.037 1` — 검증은 네 후보 모두 0.958, 소수 아래 아주 작은 계산 차이로 깊이 2가 1위

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>for문 + <code>cross_val_score</code></td><td>후보를 <strong>직접 반복</strong>, 결과 정리도 직접</td><td>후보가 1~2개, 원리를 볼 때</td></tr>
<tr><td><code>GridSearchCV</code></td><td>후보 <strong>모든 조합</strong>을 교차검증 + 최적 모델 재학습(refit)</td><td>후보가 정해져 있을 때 (기본 선택)</td></tr>
<tr><td><code>best_score_</code></td><td>최적 설정의 <strong>CV 평균</strong> (검증 점수)</td><td>설정 비교 · 보고</td></tr>
<tr><td><code>best_estimator_</code></td><td>최적 설정으로 <strong>train 전체에 다시 학습</strong>된 모델</td><td>최종 test · 실제 예측 (5-1)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 60은 `return_train_score=True`가 주석이라 셀 61의 `mean_train_score`에서 `KeyError` · 정리본은 켬

> ✅ **한 줄 정리** · `GridSearchCV(모델, 후보 dict, cv=분할기)`로 모든 후보를 교차검증하고 `best_params_` · `best_score_`로 결과를 읽는다.

**딥러닝 코드에서는**

- 딥러닝은 학습 1번이 길어 격자 전체를 돌리기 어려움 → 학습률 · 배치 크기 몇 개만 손으로 비교하거나 Optuna 같은 도구 사용

### 4-3. `B` 넓은 범위 무작위 탐색 · `RandomizedSearchCV` <sub>추가 · 원본 셀 62 GridSearch 오해</sub>

> 🎯 **후보 조합이 너무 많을 때는 어떻게 탐색하는가?**

💡 **쉽게 말하면**

- Grid는 **뷔페 음식 전부 맛보기**, Random은 **무작위로 20접시만 맛보기**.
- 접시가 많을수록 Random이 시간 대비 효율적.

**실행 흐름**

1. 깊이 9 × 최소 샘플 18 = 162가지
2. Grid 전부 vs Random 20개
3. 최적 설정 · 점수 비교

**먼저 예측** · 162가지 중 20가지만 무작위로 시험하면 최고 점수는 Grid보다 많이 낮을까? — 답은 코드 실행 결과 아래에

In [13]:
from scipy.stats import randint                         # randint(a, b) → a ~ b-1 정수 분포
from sklearn.model_selection import RandomizedSearchCV  # 무작위 탐색기


# 넓은 탐색 범위: 깊이 1~9 × 나누기 최소 샘플 2~19 = 162가지
# 왜 · 후보가 많을 때 전부 시험하는 비용을 실감하기 위해
space = {"max_depth": list(range(1, 10)),  # 9 × 18 = 162가지
         "min_samples_split": list(range(2, 20))}


# GridSearchCV → 162가지 전부 시험
# 왜 · 전수 탐색의 결과를 기준으로 무작위 탐색과 비교하기 위해
grid_all = GridSearchCV(tree, space, cv=skf).fit(X_train, y_train)  # 전부 시험
n_grid = len(grid_all.cv_results_["params"])                        # 시험한 후보 수
print(f"Grid   : 후보 {n_grid}개 · 최적 {grid_all.best_params_}"    # Grid 결과
      f" · CV {grid_all.best_score_:.3f}")


# RandomizedSearchCV → 같은 범위에서 20가지만 무작위로 뽑아 시험
# 왜 · 적은 시도로도 비슷한 설정을 찾을 수 있는지 확인하기 위해
dist = {"max_depth": randint(1, 10), "min_samples_split": randint(2, 20)}  # 같은 범위의 분포
rand = RandomizedSearchCV(tree, dist, n_iter=20, cv=skf,                   # 20가지만 뽑음
                          random_state=SEED)                               # n_iter = 뽑을 후보 수
rand.fit(X_train, y_train)                                                 # 20 × 5 = 100번 학습
n_rand = len(rand.cv_results_["params"])                                   # 시험한 후보 수
print(f"Random : 후보 {n_rand}개 · 최적 {rand.best_params_}"               # Random 결과
      f" · CV {rand.best_score_:.3f}")

Grid   : 후보 162개 · 최적 {'max_depth': 2, 'min_samples_split': 2} · CV 0.958
Random : 후보 20개 · 최적 {'max_depth': 2, 'min_samples_split': 16} · CV 0.958


**결과** · 예측 정답은 같음 — 둘 다 `0.958` (학습 횟수는 810번 vs 100번)

- `Grid   : 후보 162개 · 최적 {'max_depth': 2, 'min_samples_split': 2} · CV 0.958` — 162 × 5 = 810번 학습
- `Random : 후보 20개 · 최적 {'max_depth': 2, 'min_samples_split': 16} · CV 0.958` — 20 × 5 = 100번으로 같은 점수

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>GridSearchCV</code></td><td><strong>모든 조합</strong> 시험 → 빠짐없지만 조합 수만큼 느려짐</td><td>후보가 적을 때 (조합 수십 개 이하)</td></tr>
<tr><td><code>RandomizedSearchCV</code></td><td>범위에서 <strong>n_iter개만 무작위</strong> → 시간은 고정, 최적을 놓칠 수 있음</td><td>후보 범위가 넓을 때 · 감을 잡는 1차 탐색</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 둘 다 **내가 준 범위 안에서만** 찾음
> - 범위를 잘못 잡으면 아무리 오래 돌려도 좋은 설정은 나오지 않음

> ✅ **한 줄 정리** · 후보가 적으면 `GridSearchCV`, 범위가 넓으면 `RandomizedSearchCV`로 `n_iter`개만 뽑아 탐색한다.

**딥러닝 코드에서는**

- 학습률은 `loguniform(1e-5, 1e-1)`처럼 자릿수 단위로 무작위 탐색하는 것이 표준

---
## 5. 최종 테스트 · 딥러닝 연결

> **이 장의 흐름**
>
> - 4장까지 train 안에서만 고르기를 끝냈다
> - 봉인한 test를 언제 · 몇 번 여는지가 마지막 규칙.
> - 딥러닝은 교차검증 대신 val을 따로 떼는 3분할을 씀

**파이프라인 위치** · ⑦ 최종 보고 — test 1회, 딥러닝의 train · val · test

### 5-1. `A` 최종 test 1회 · `best_estimator_` <sub>원본 02 셀 63 ~ 65</sub>

> 🎯 **최적 모델은 무엇으로 학습된 상태이며, 봉인한 test는 언제 몇 번 쓰는가?**

💡 **쉽게 말하면**

- 모의고사(CV)로 고른 학생에게 **봉인해 둔 수능(test)을 딱 한 번**.
- 결과를 보고 다시 공부시키면 수능이 모의고사가 됨.

**실행 흐름**

1. best_estimator_ 꺼내기
2. 학습 샘플 수 확인
3. test로 1회 채점
4. CV 점수와 비교

**먼저 예측** · 최적 모델은 fold의 96개로 학습된 상태일까, train 120개 전체로 학습된 상태일까? — 답은 코드 실행 결과 아래에

In [14]:
# 최적 모델 꺼내기: refit=True(기본) → X_train 전체에 다시 학습된 모델
# 왜 · 고른 설정으로 train 전체에 다시 학습된 모델을 쓰기 위해
best_model = grid.best_estimator_            # refit된 최적 모델
print("최적 모델:", best_model)              # 고른 설정 확인
n_used = best_model.tree_.n_node_samples[0]  # 트리 맨 위 칸 = 학습 샘플 전체
print("학습에 쓴 샘플 수:", n_used)          # 120이면 train 전체


# 봉인했던 test로 단 1회 채점
# 왜 · 선택에 쓰지 않은 test로 한 번만 채점해 공정한 최종 점수를 얻기 위해
test_pred = best_model.predict(X_test)        # 봉인했던 test 처음 사용
test_acc = accuracy_score(y_test, test_pred)  # 최종 점수 (1회)
# 고를 때 점수와 비교
print(f"교차검증 점수 {grid.best_score_:.3f} | 최종 test 점수 {test_acc:.3f}")
print(f"test에서 틀린 개수: {(test_pred != y_test).sum()}개 / {len(y_test)}개")  # 틀린 수

최적 모델: DecisionTreeClassifier(max_depth=2, random_state=18)
학습에 쓴 샘플 수: 120
교차검증 점수 0.958 | 최종 test 점수 0.900
test에서 틀린 개수: 3개 / 30개


**결과** · 예측 정답은 train `120`개 전체 — `refit=True`(기본)가 최적 설정으로 다시 학습

- `최적 모델: DecisionTreeClassifier(max_depth=2, random_state=18)` — 4-2에서 고른 설정
- `학습에 쓴 샘플 수: 120` — fold의 96개가 아니라 train 전체
- `교차검증 점수 0.958 | 최종 test 점수 0.900` — test가 5.8%p 낮음 → 이것이 처음 보는 데이터에서의 현실
- `test에서 틀린 개수: 3개 / 30개` — test 30송이 중 27송이 정답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>grid.best_score_</code></td><td>train 안에서 <strong>고를 때 쓴</strong> 점수 (검증)</td><td>설정 선택</td></tr>
<tr><td>최종 test 점수</td><td>고르는 데 <strong>한 번도 안 쓴</strong> 데이터 점수</td><td>보고 · 배포 전 확인 (1회)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - test 점수가 낮다고 설정을 바꿔 다시 test로 확인하기
> - 그 순간 test가 검증이 되어 점수를 믿을 수 없음 (5-3)

> ✅ **한 줄 정리** · 모델 선택은 CV로 끝내고, `best_estimator_`로 봉인한 test를 마지막에 한 번만 채점해 보고한다.

**딥러닝 코드에서는**

- 학습이 끝난 뒤 `model.load_state_dict(torch.load("best.pt"))` → `evaluate(model, test_loader)` 한 번

### 5-2. `A` train · val · test 3분할 · `train · val · test` <sub>추가 · 원본 셀 63 역할 분담 그림</sub>

> 🎯 **교차검증 대신 검증 데이터를 따로 떼어 두는 3분할은 어떻게 만들고 왜 쓰는가?**

💡 **쉽게 말하면**

- 교과서(train) · **모의고사지 한 묶음**(val) · 수능(test).
- 딥러닝은 공부 1번이 너무 길어 모의고사를 5번 볼 수 없어서 모의고사지를 따로 떼어 둠.

**실행 흐름**

1. test 20% 떼기
2. 남은 80%를 75 : 25로
3. 60 : 20 : 20
4. 깊이 후보를 val로 채점
5. 고른 설정으로 test 1회

**먼저 예측** · 두 번 나눠서 60 : 20 : 20을 만들려면 두 번째 `test_size`는 0.2일까? — 답은 코드 실행 결과 아래에

In [15]:
# 3분할: test 20%를 먼저 떼고 → 남은 80%를 train 75% · val 25%로
# 왜 · 딥러닝처럼 교차검증이 무거울 때 쓰는 train · val · test 3분할을 익히기 위해
X_rest, X_te3, y_rest, y_te3 = train_test_split(                      # test 20% 먼저 봉인
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=SEED)
X_tr3, X_val3, y_tr3, y_val3 = train_test_split(                      # 남은 80%의 25% = 전체 20%
    X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=SEED)
print(f"train {len(X_tr3)} | val {len(X_val3)} | test {len(X_te3)}")  # 60 : 20 : 20


# 검증으로 고르기: 깊이 후보마다 train으로 학습 → val로 채점
# 왜 · 설정은 val로만 고르고 test는 끝까지 아껴 두기 위해
val_scores = {}                                                     # 깊이별 val 점수
for d in [1, 2, 3, 4, None]:                                        # 후보마다
    model = DecisionTreeClassifier(max_depth=d, random_state=SEED)  # 깊이만 바꿈
    val_scores[d] = model.fit(X_tr3, y_tr3).score(X_val3, y_val3)   # train 학습 → val 채점
best_d = max(val_scores, key=val_scores.get)                        # 값이 가장 큰 키 (동점이면 앞)
rounded = {d: round(v, 3) for d, v in val_scores.items()}           # 보기 좋게 반올림
print("val 점수:", rounded, "→ 선택 max_depth =", best_d)           # 고른 깊이


# 고른 설정으로 test 1회
# 왜 · val로 고른 설정 그대로 test를 한 번만 채점하기 위해
final = DecisionTreeClassifier(max_depth=best_d, random_state=SEED)  # 고른 설정으로
final.fit(X_tr3, y_tr3)                                              # train으로 학습
print(f"val 점수 {val_scores[best_d]:.3f}"                           # val vs test 1회
      f" | 최종 test 점수 {final.score(X_te3, y_te3):.3f}")

train 90 | val 30 | test 30
val 점수: {1: 0.667, 2: 0.967, 3: 0.933, 4: 0.933, None: 0.933} → 선택 max_depth = 2
val 점수 0.967 | 최종 test 점수 0.900


**결과** · 예측 정답은 `0.25` — 남은 80%의 25%가 전체의 20%

- `train 90 | val 30 | test 30` — 전체를 60 : 20 : 20으로
- `val 점수: {1: 0.667, 2: 0.967, 3: 0.933, 4: 0.933, None: 0.933} → 선택 max_depth = 2` — val이 가장 높은 깊이 선택
- `val 점수 0.967 | 최종 test 점수 0.900` — 고를 때 쓴 val 점수는 낙관적 → test가 더 낮음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>train · test 2분할 + 교차검증</td><td>검증을 <strong>train 안에서 K번</strong> 돌려 만듦</td><td>데이터가 적고 학습이 빠를 때 (머신러닝)</td></tr>
<tr><td>train · val · test 3분할</td><td>검증용을 <strong>따로 한 덩어리</strong> 떼어 둠</td><td>학습이 오래 걸릴 때 (딥러닝 기본)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 두 번째 분할의 `test_size`는 **남은 데이터 기준** 비율, `stratify`도 두 번째에는 `y_rest`로

> ✅ **한 줄 정리** · train으로 학습하고 val로 설정을 고른 뒤, test는 마지막에 한 번만 채점한다.

**딥러닝 코드에서는**

- PyTorch 표준 흐름
- `random_split(dataset, [0.6, 0.2, 0.2], generator=torch.Generator().manual_seed(42))` → epoch마다 val로 평가해 `best.pt` 저장 → 끝나고 test 1회

### 5-3. `A` 검증과 최종 테스트의 경계 · `val vs test` <sub>추가</sub>

> 🎯 **test로 여러 번 채점해서 가장 좋은 모델을 고르면 왜 안 되는가?**

💡 **쉽게 말하면**

- 수능 문제를 미리 50번 풀어 보고 제일 잘 본 날의 점수를 내는 것.
- **실력이 없어도** 50번 중 운 좋은 1번은 높게 나옴.

<img src="images/ml1_peek.png" alt="test로 고르면 부풀려지는 점수" width="587">

**실행 흐름**

1. 엉터리 정답으로 3분할
2. 후보 50개
3. val로 고르기 vs test로 고르기
4. 찍기 수준 1/3과 비교

**먼저 예측** · 배울 규칙이 없는 데이터에서 test 점수가 가장 높은 후보를 고르면 test 점수는 찍기 수준(0.333)일까? — 답은 코드 실행 결과 아래에

In [16]:
# 배울 규칙이 없는 엉터리 정답(1-2의 y_shuffled)으로 3분할 → 실력 = 찍기 1/3
# 왜 · test로 여러 번 고르면 실력 없이도 점수가 부풀려지는 것을 보여 주기 위해
X_rest, X_te_f, y_rest, y_te_f = train_test_split(    # 엉터리 정답으로 test 떼기
    X_full, y_shuffled, test_size=0.2, random_state=SEED)
X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(  # 남은 것을 train · val로
    X_rest, y_rest, test_size=0.25, random_state=SEED)


# 후보 모델 50개 (깊이 3 · 특성 2개씩만 보는 트리, seed만 다름)
# 왜 · seed만 다른 후보를 많이 만들어 우연히 좋은 점수가 나오는 상황을 재현하기 위해
cands = []                                                                   # 후보 모델 50개
for s in range(50):                                                          # seed만 다른 50개
    m = DecisionTreeClassifier(max_depth=3, max_features=2, random_state=s)  # 특성 2개씩만
    cands.append(m.fit(X_tr_f, y_tr_f))                                      # 학습해서 저장
val_acc = np.array([m.score(X_val_f, y_val_f) for m in cands])               # 50개의 val 점수
test_acc = np.array([m.score(X_te_f, y_te_f) for m in cands])                # 50개의 test 점수


# 올바른 방법: val로 고르고 test는 1회 → 찍기 수준이 그대로 드러남
# 왜 · 올바른 절차에서는 찍기 수준의 실력이 그대로 드러난다는 것을 확인하기 위해
i_val = val_acc.argmax()                             # val 최고 후보의 번호
print(f"val로 고른 모델 → val {val_acc[i_val]:.3f}"  # 올바른 방법의 결과
      f" · test {test_acc[i_val]:.3f}")


# 잘못된 방법: test로 50번 채점해 최고를 고름 → 실력 없이도 점수가 부풀려짐
# 왜 · test를 선택에 쓰면 점수가 거짓으로 높아진다는 것을 비교하기 위해
print(f"test로 고른 모델 → test {test_acc.max():.3f}"  # 잘못된 방법의 결과
      f" (찍기 수준 {1 / 3:.3f})")
print(f"test 점수 50개 평균 {test_acc.mean():.3f}")    # 평균은 찍기 근처

val로 고른 모델 → val 0.300 · test 0.400
test로 고른 모델 → test 0.467 (찍기 수준 0.333)
test 점수 50개 평균 0.386


**결과** · 예측 정답은 아님 — `0.467`까지 부풀려짐 (val로 고르면 `0.400`)

- `val로 고른 모델 → val 0.300 · test 0.400` — val 최고점도 찍기 수준 → 규칙이 없다는 사실이 보임
- `test로 고른 모델 → test 0.467 (찍기 수준 0.333)` — 실력이 없는데 13.4%p 부풀려짐
- `test 점수 50개 평균 0.386` — 평균은 찍기 근처 → 최고점만 골라서 생긴 착시

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>검증 데이터 (val · CV)</td><td><strong>여러 번</strong> 보면서 모델 · 설정 · epoch를 고름</td><td>개발 중 계속</td></tr>
<tr><td>최종 테스트 (test)</td><td><strong>마지막에 1번</strong>만, 결과를 보고 다시 고치지 않음</td><td>보고 · 제출 직전</td></tr>
<tr><td>test로 여러 번 고르기</td><td>test가 사실상 검증이 됨 → <strong>점수가 부풀려짐</strong></td><td>쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - "test 점수 보고 조금만 고쳐서 다시"도 같은 행동
> - 횟수가 쌓일수록 test가 검증으로 변함

> ✅ **한 줄 정리** · 고르는 일은 검증 데이터로만 하고, 최종 테스트는 결과를 보고 다시 고치지 않을 마지막 한 번에만 쓴다.

**딥러닝 코드에서는**

- Kaggle의 public 리더보드에 수십 번 제출해 맞춘 모델이 private 리더보드에서 순위가 급락하는 현상이 바로 이것

---
## 6. 종합 실습 · 펭귄 품종 분류를 처음부터 끝까지

> **이 장의 흐름**
>
> - 1~5장을 새 데이터 한 개에 순서대로 적용.
> - 수업 2-1 펭귄 실습(문제 Q1 ~ Q12 · 정답)과 같은 흐름이며, 2-1 실습의 설정(`random_state=7` 분할 · 모델 seed 42)을 재료로 씀

**파이프라인 위치** · ① 데이터 준비 → ② 나누기 → ⑥ 비교 · 탐색 → ⑦ 최종 test 1회

> 🎯 **새 데이터에서 분할 → 교차검증 비교 → 설정 탐색 → 최종 test 1회를 어떤 순서로 하는가?**

- 💡 **쉽게 말하면** 1~5장은 붓꽃으로 하나씩 배운 것이고, 이번엔 **펭귄으로 처음부터 혼자 끝까지**.
- 순서를 지키는 것이 핵심 — test는 맨 처음에 봉인하고 맨 마지막에 한 번만 엶.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 섹션</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>seaborn 펭귄 불러오기 · 측정값 결측 제거 · 특성 단위 확인</td><td>0장</td></tr>
<tr><td>1</td><td>test 20% 봉인 (stratify)</td><td>2-1</td></tr>
<tr><td>2</td><td>train 안에서 5-Fold로 트리 vs KNN 비교</td><td>3-1 · 3-3 · 3-5</td></tr>
<tr><td>3</td><td>이긴 트리의 깊이를 GridSearchCV로 탐색 · 학습 vs 검증</td><td>4-1 · 4-2</td></tr>
<tr><td>4</td><td>최적 모델(refit)로 test 1회</td><td>5-1</td></tr>
</tbody></table>

**먼저 예측** · 측정값을 그대로 넣으면 트리와 KNN 중 어느 쪽이 교차검증에서 크게 질까? — 답은 코드 실행 결과 아래에

**실행 흐름**

1. 펭귄 데이터 읽기 · 측정값 결측 제거
2. test 20% 봉인 (2-1 실습과 같은 분할)
3. train 안에서만 5-Fold로 모델 비교

In [17]:
import seaborn as sns  # 예제 데이터 불러오기용 (2-1 실습과 같은 방법)


# 0단계: 펭귄 데이터 읽기 (2-1 실습과 같은 준비 · 측정값 결측 2마리 제거)
# 왜 · 앞 실습과 같은 데이터 준비로 결과를 이어서 비교하기 위해
peng_raw = sns.load_dataset("penguins")                          # 344마리 표 (DataFrame)
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",  # 측정값 4열
        "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)       # 측정값이 빈 2마리 제거
X_p = peng[meas].to_numpy()                                      # 표 → 배열
y_p = peng["species"].to_numpy()                                 # 정답 = 품종 이름(문자열) 그대로
print(f"펭귄 {len(peng_raw)} → {len(peng)}마리 · 특성 {X_p.shape[1]}개"  # 정리 결과
      f" · 품종 {sorted(set(y_p))}")
print("특성별 최댓값:", X_p.max(axis=0))                         # 몸무게만 단위가 수천


# 1단계: test 20% 봉인 (stratify · 2-1 실습과 같은 random_state=7)
# 왜 · 앞 실습과 같은 분할로 test를 먼저 떼어 끝까지 아끼기 위해
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(            # test 20% 봉인
    X_p, y_p, test_size=0.2, stratify=y_p, random_state=7)
print(f"train {len(Xp_tr)}마리 | test {len(Xp_te)}마리")  # 273 · 69


# 2단계: train 안에서만 5-Fold로 모델 비교
# 왜 · test를 보지 않고 모델을 고르기 위해
cv_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)        # 비교용 5등분
peng_models = {"DecisionTree": DecisionTreeClassifier(random_state=42),  # 후보 2개
               "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)}
for name, m in peng_models.items():                                      # 모델마다
    s = cross_val_score(m, Xp_tr, yp_tr, cv=cv_p)                        # train 안에서만
    print(f"{name:<13} CV {s.mean():.3f} ± {s.std():.3f}")               # 평균 ± 흔들림

펭귄 344 → 342마리 · 특성 4개 · 품종 ['Adelie', 'Chinstrap', 'Gentoo']
특성별 최댓값: [  59.6   21.5  231.  6300. ]
train 273마리 | test 69마리
DecisionTree  CV 0.960 ± 0.022
KNN (k=5)     CV 0.784 ± 0.023


**실행 흐름**

1. 이긴 트리의 깊이를 GridSearchCV로 탐색 (train 안)
2. 최적 모델로 test 1회 · 출력

In [18]:
# 3단계: 이긴 트리의 깊이를 GridSearchCV로 탐색 (train 안에서만)
# 왜 · 고른 모델의 설정도 train 안에서만 정하기 위해
grid_p = GridSearchCV(                                                 # 이긴 트리의 깊이 탐색
    DecisionTreeClassifier(random_state=42),
    {"max_depth": [1, 2, 3, 4, None]}, cv=cv_p, return_train_score=True)
grid_p.fit(Xp_tr, yp_tr)                                               # train 안에서만
cols_p = {"param_max_depth": "max_depth", "mean_train_score": "학습",  # 보여 줄 열
          "mean_test_score": "검증"}
res_p = pd.DataFrame(grid_p.cv_results_)[list(cols_p)].rename(columns=cols_p)  # 결과 → 표
print(res_p.round(3).to_string(index=False))                           # 깊이별 학습 · 검증
print("최적 깊이:", grid_p.best_params_,                               # 고른 깊이
      "| CV:", round(grid_p.best_score_, 3))


# 4단계: 최적 모델로 test 1회 (refit → train 273마리 전체로 학습된 모델)
# 왜 · 선택이 끝난 모델로 test를 한 번만 채점하기 위해
final_p = grid_p.best_estimator_                            # train 전체로 refit된 모델
n_ok = (final_p.predict(Xp_te) == yp_te).sum()              # 맞힌 마리 수
print(f"최종 test 점수: {final_p.score(Xp_te, yp_te):.3f}"  # test 1회
      f" ({len(yp_te)}마리 중 {n_ok}마리 정답)")

max_depth    학습    검증
        1 0.788 0.762
        2 0.967 0.941
        3 0.983 0.941
        4 0.999 0.963
     None 1.000 0.960
최적 깊이: {'max_depth': 4} | CV: 0.963
최종 test 점수: 0.913 (69마리 중 63마리 정답)


**결과** · 예측 정답은 KNN (CV `0.784`) — 몸무게(최대 6300)가 부리 길이의 100배라 거리가 몸무게만 봄

- `펭귄 344 → 342마리 · 특성 4개` — 측정값이 빈 2마리 제거 (2-1 실습과 같은 342마리)
- `특성별 최댓값: [  59.6   21.5  231.  6300. ]` — 몸무게만 단위가 수천 → KNN이 불리한 이유
- `train 273마리 | test 69마리` — 342 × 0.2 → 69마리 봉인
- `DecisionTree  CV 0.960 ± 0.022` — 트리는 특성 하나씩 기준을 정해 단위 차이에 영향 없음
- `KNN (k=5)     CV 0.784 ± 0.023` — 거리가 몸무게에 휘둘림 → 스케일링으로 해결 (머신러닝-2)
- `1 0.788 0.762` / `4 0.999 0.963` — 깊이 1은 과소적합 · 깊이 4가 검증 최고
- `최적 깊이: {'max_depth': 4} | CV: 0.963` — 2-1 실습 정답과 같은 깊이 4
- `최종 test 점수: 0.913 (69마리 중 63마리 정답)` — CV보다 5%p 낮음 · 2-1 정답의 test 0.913과 같음

<img src="images/ml1_penguin.png" alt="펭귄 종합 실습 결과" width="624">

**해석**

- CV 0.963 → test 0.913은 5-1 · 5-2와 같은 패턴 (고를 때 쓴 점수가 더 낙관적)
- 이 차이를 보고 다시 고치지 않는 것이 규칙 · 개선은 **train 안에서** 함
- 머신러닝-2에서 스케일링 · 파이프라인으로 KNN · 로지스틱을 다시 살림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>2-1 실습 정답의 <code>cv=5</code></td><td>섞지 않는 5등분 → CV 0.9708</td><td>데이터가 이미 섞여 있을 때</td></tr>
<tr><td>정리본의 <code>cv_p</code> (섞기 · seed 42)</td><td>섞은 5등분 → CV 0.963</td><td>모델 · 설정 비교를 같은 분할로 재현할 때 (기본 선택)</td></tr>
<tr><td>트리</td><td>특성마다 따로 기준 → <strong>단위 차이에 강함</strong></td><td>전처리 없이 빠른 기준선</td></tr>
<tr><td>KNN · 로지스틱</td><td>거리 · 가중합 → <strong>단위가 큰 특성이 지배</strong></td><td>스케일링한 뒤 (머신러닝-2)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - test 점수가 CV보다 낮다고 이 자리에서 깊이를 바꿔 test로 다시 확인하지 않기
> - 그렇게 하면 5-3의 부풀림

> ✅ **한 줄 정리** · test를 맨 처음 봉인하고, train 안에서만 교차검증으로 비교 · 탐색한 뒤, 고른 모델 하나로 test를 마지막에 한 번 채점한다.

**딥러닝 코드에서는**

- `random_split`으로 test 봉인
- train · val로 epoch마다 평가하며 `best.pt` 저장
- 학습이 끝난 뒤 `best.pt`를 불러 test 1회

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 학습 점수의 착시</td><td>엉터리 정답도 학습 데이터에선 0.993, test에선 0.233 → 실력은 처음 보는 데이터로만</td><td>1-2 · 2-1</td></tr>
<tr><td>2. 분할 · 분할 운</td><td>stratify로 10 · 10 · 10, 분할 seed만 바꿔도 0.900 ~ 0.967</td><td>2-1 · 2-2</td></tr>
<tr><td>3. 교차검증 도구</td><td>for문 5-Fold = <code>cross_val_score</code> (0.958), 지표 여러 개는 <code>cross_validate</code></td><td>3-1 · 3-3 · 3-4</td></tr>
<tr><td>4. 설정 탐색 · 과적합</td><td>깊이 4부터 학습만 1.000 · 검증 0.958 → 동점이면 깊이 2, Grid 810번 = Random 100번 점수</td><td>4-1 ~ 4-3</td></tr>
<tr><td>5. 3분할 · test 1회</td><td>60 : 20 : 20을 두 번 나눠 만들고, test로 고르면 0.333 → 0.467로 부풀려짐</td><td>5-2 · 5-3</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>모델 3단계</td><td>만들기 → <code>fit</code> → <code>predict</code> / <code>score</code></td><td>학습 루프 = <code>fit</code>, <code>model(x)</code> = <code>predict</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>착시</td><td>학습한 데이터로 낸 점수 = 외운 양</td><td><code>train_acc</code>만 보고 판단하지 않기</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>분할</td><td><code>train_test_split(stratify=y, random_state=고정)</code></td><td><code>random_split(..., generator=seed)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>분할 운</td><td>test 점수 1개에는 운이 섞임</td><td>seed 여러 개 평균</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>K-Fold</td><td>train 안에서 K등분, 1등분씩 검증 → mean ± std</td><td>fold마다 새 모델 · <code>Subset</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>나누는 방법</td><td>분류 = <code>StratifiedKFold(shuffle=True)</code></td><td><code>DataLoader(shuffle=True)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>한 줄 교차검증</td><td><code>cross_val_score(모델, X, y, cv=분할기)</code></td><td>(직접 for문)</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>모델 선택</td><td>같은 분할 · mean 우선 · 비슷하면 std</td><td>같은 val · 같은 seed</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>과적합</td><td>학습만 높고 검증과 차이가 큼</td><td><code>val_loss</code>가 다시 오르는 epoch</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>GridSearchCV</td><td>모든 조합 CV + refit → <code>best_estimator_</code></td><td>학습률 · 배치 크기 비교</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>최종 test</td><td>모든 선택이 끝난 뒤 1회</td><td><code>best.pt</code> → test 1회</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>3분할</td><td>test 먼저 → 남은 것에서 val (<code>0.25</code>)</td><td>train · val · test 기본 구조</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>검증 vs 테스트</td><td>고르기는 val, test로 고르면 부풀려짐</td><td>리더보드 과적합</td></tr>
<tr><td style="text-align: center;"><code>B</code></td><td>cross_validate</td><td>지표 여러 개 · 학습 점수, <code>test_</code> = fold 검증</td><td><code>history</code> 기록</td></tr>
<tr><td style="text-align: center;"><code>B</code></td><td>RandomizedSearchCV</td><td>넓은 범위에서 <code>n_iter</code>개</td><td>학습률 로그 스케일 탐색</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 모델을 쓰는 3단계와, `score`와 `accuracy_score`의 관계는?
- **1-2** · 학습한 데이터로 채점하면 왜 실력을 알 수 없는가?
- **2-1** · `stratify`와 `random_state`는 왜 넣는가?
- **2-2** · 한 번 나눈 test 점수를 그대로 믿으면 안 되는 이유는?
- **3-1** · 5-Fold 교차검증은 몇 번 학습하고, 검증에는 무엇을 쓰는가?
- **3-2** · 정렬된 데이터를 섞지 않은 `KFold`로 나누면 무슨 일이 생기는가?
- **3-3** · `cv=5`와 `cv=StratifiedKFold(5, shuffle=True, random_state=…)`의 차이는?
- **3-4** · `cross_validate` 결과의 `test_accuracy`는 최종 test 점수인가?
- **3-5** · CV mean이 비슷한 두 모델 중 무엇을 고르는가?
- **4-1** · 과소적합과 과적합을 학습 · 검증 점수로 어떻게 구분하는가?
- **4-2** · `best_score_`와 `best_estimator_`는 각각 무엇인가?
- **4-3** · `GridSearchCV`와 `RandomizedSearchCV`는 언제 쓰는가?
- **5-2** · 60 : 20 : 20 3분할을 `train_test_split` 두 번으로 만드는 방법은?
- **5-3** · test로 여러 번 채점해 모델을 고르면 왜 안 되는가?
- **6** · 새 데이터에서 분할부터 최종 test까지의 순서는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

- 원본 model_selection의 **실습 미션 1 ~ 4**(빈칸 72 = 미션 3 결과표)를 그대로 가져온 문제.
- 원본은 "정확도를 기록"이라고만 했지만, 5-3의 규칙대로 **test가 아니라 X_train의 교차검증(`skf`) 평균**으로 기록.
- 아래 빈 셀에 직접 작성 후 실행.

1. DecisionTree의 `max_depth`를 **2 · 3 · None**으로 바꿔 CV 평균 정확도 기록
2. KNN의 `n_neighbors`를 **3 · 5 · 9**로 바꿔 CV 평균 정확도 기록
3. 1 · 2의 결과를 `model` · `param` · `accuracy` 세 열의 **표**(DataFrame)로 만들고 가장 좋은 설정 찾기
4. (도전) GridSearchCV 후보에 `min_samples_split=[2, 4, 8]`을 추가해 다시 실행하고 **상위 5개** 점수 정리

In [19]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-1의 분할을 `test_size=0.3`으로 바꾸면 train · test는 몇 송이이고, test의 클래스별 개수는? (stratify 유지)
2. 3-1의 `StratifiedKFold`를 `n_splits=4`로 바꾸면 fold마다 학습 · 검증은 몇 개씩일까?
3. 4-2의 후보에 `criterion=["gini", "entropy"]`를 추가하면 후보는 몇 개이고, refit까지 학습은 몇 번 하나? 예측 → 실행

In [20]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 모델 만들기 → `fit(X, y)` → `predict(X)` → 채점. `score`는 안에서 `predict` 후 `accuracy_score`를 계산해 같은 값
- **1-2** · 모델이 답을 외웠는지 배웠는지 구분이 안 됨 (정답을 무작위로 섞어도 0.993)
- **2-1** · `stratify`는 train · test의 클래스 비율 유지, `random_state`는 같은 버전에서 같은 분할로 재현
- **2-2** · 분할 운이 섞임 (seed만 바꿔도 0.900 ~ 0.967)
- **3-1** · 5번 학습. 매번 train의 1/5을 검증, 나머지 4/5로 학습. test는 쓰지 않음
- **3-2** · 검증 fold에 학습에 없는 클래스만 들어가 0점이 나옴 → `shuffle=True` 또는 `StratifiedKFold`
- **3-3** · `cv=5`는 섞지 않는 StratifiedKFold(5), 객체를 넘기면 섞기 · seed까지 고정
- **3-4** · 아님. 각 fold의 검증 점수 (이름만 test)
- **3-5** · 차이가 1~2%p 이내면 std가 작은 쪽, 분할 seed를 바꿔 순위도 재확인
- **4-1** · 둘 다 낮으면 과소적합, 학습만 높고 차이가 크면 과적합
- **4-2** · `best_score_` = 최적 설정의 CV 평균, `best_estimator_` = 최적 설정으로 train 전체에 다시 학습된 모델
- **4-3** · 후보가 적으면 Grid(전부), 범위가 넓으면 Random(n_iter개)
- **5-2** · test 0.2를 먼저 떼고, 남은 데이터에서 `test_size=0.25` (= 전체의 20%)
- **5-3** · test가 고르는 데 쓰여 검증이 되고, 최고점만 골라 점수가 부풀려짐 (찍기 0.333 → 0.467)
- **6** · test 봉인 → train 안에서 CV 비교 → GridSearchCV 탐색 → `best_estimator_`로 test 1회

#### 코드 문제 답

```python
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV)

X_full, y_full = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=18)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=18)

# 미션 1 · 2. 설정마다 같은 5등분으로 CV 평균 (test는 쓰지 않음)
rows = []
for d in [2, 3, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=18)
    s = cross_val_score(m, X_train, y_train, cv=skf)
    rows.append(("DecisionTree", f"max_depth={d}", round(s.mean(), 3)))
for k in [3, 5, 9]:
    m = KNeighborsClassifier(n_neighbors=k)
    s = cross_val_score(m, X_train, y_train, cv=skf)
    rows.append(("KNN", f"n_neighbors={k}", round(s.mean(), 3)))
```

```python
# 미션 3. 결과 표 → 가장 좋은 설정
table = pd.DataFrame(rows, columns=["model", "param", "accuracy"])
acc = table["accuracy"].tolist()
best = table.loc[table["accuracy"].idxmax(), "param"]
print(acc)  # [0.958, 0.958, 0.958, 0.975, 0.983, 0.958]
print(best)  # n_neighbors=5

# 미션 4. 후보 4 × 3 = 12개 → 상위 5개
space = {"max_depth": [2, 3, 4, None], "min_samples_split": [2, 4, 8]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=18), space, cv=skf)
grid.fit(X_train, y_train)
res = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
top5 = res["mean_test_score"].head(5).round(3).tolist()
print(len(res))  # 12
print(grid.best_params_)  # {'max_depth': 2, 'min_samples_split': 2}
print(top5)  # [0.958, 0.958, 0.958, 0.958, 0.958]
```

**미션 3 결과표 코멘트**

- 트리는 깊이를 바꿔도 0.958로 같음(4-1과 같은 이유), KNN은 k=5가 0.983으로 가장 좋음.
- 미션 4는 상위 5개가 모두 0.958 동점
- 붓꽃은 너무 쉬워 설정 차이가 드러나지 않음, 동점이면 가장 단순한 설정(깊이 2)

#### 변형 문제 답

*문제 → 답*

- **1** · 150 × 0.3 = 45 → train 105 · test 45, 클래스별 `[15 15 15]`
- **2** · 120 ÷ 4 = 30 → 학습 90 · 검증 30 (4번)
- **3** · 4 × 2 = 8개 · 8 × 5 + 1 = 41번 (최적은 gini · 깊이 2 · CV 0.958로 그대로)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) test 점수를 보고 깊이를 다시 고름</td><td>test를 검증처럼 씀</td><td>고르기는 CV · val, test는 마지막 1회</td><td>이 숫자로 무엇을 고르고 있나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 학습 데이터와 테스트 데이터를 왜 나누나요?**
> - 학습에 쓴 데이터로 채점하면 모델이 규칙을 배운 것인지 답을 외운 것인지 구분할 수 없습니다.
> - 실제로 정답을 무작위로 섞어도 트리는 학습 데이터에서 99% 이상 맞힙니다.
> - 그래서 처음 보는 데이터를 test로 떼어 봉인해 두고, 그 점수로 새 데이터에서의 성능을 추정합니다.

**Q2. 교차검증은 왜 하고, StratifiedKFold는 언제 쓰나요?**
> - 한 번 나눈 검증 점수는 어떤 샘플이 검증에 들어갔는지에 따라 흔들립니다.
> - K-Fold는 train을 K등분해 돌아가며 검증하고 평균과 표준편차로 실력을 봅니다.
> - 분류 문제에서는 fold마다 클래스 비율을 맞추는 StratifiedKFold를 쓰고, 정렬된 데이터라면 반드시 섞습니다.

**Q3. GridSearchCV와 RandomizedSearchCV의 차이는?**
> - GridSearchCV는 후보의 모든 조합을 교차검증해 빠짐없지만 조합 수만큼 느려집니다.
> - RandomizedSearchCV는 범위에서 정해진 개수만 무작위로 뽑아 시간이 고정되고, 넓은 범위를 탐색할 때 효율적입니다.
> - 둘 다 내가 준 범위 안에서만 찾기 때문에 범위 설계가 먼저입니다.

**Q4. 검증 데이터와 테스트 데이터는 어떻게 다른가요?**
> - 검증 데이터는 모델과 하이퍼파라미터, 딥러닝에서는 저장할 epoch를 고르는 데 여러 번 씁니다.
> - 테스트 데이터는 모든 선택이 끝난 뒤 한 번만 채점합니다.
> - 테스트 점수를 보고 다시 고치면 테스트가 검증이 되어 점수가 낙관적으로 부풀려지므로, 최종 성능으로 보고할 수 없습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>DecisionTreeClassifier</code> · <code>LogisticRegression</code> · <code>KNeighborsClassifier</code></td><td>sklearn.tree · linear_model · neighbors</td><td>분류 모델 3종</td><td>1-1 <code>clf = DecisionTreeClassifier(random_state=SEED)</code></td></tr>
<tr><td><code>fit()</code> · <code>predict()</code> · <code>score()</code></td><td>모든 sklearn 모델 (Estimator)</td><td>학습 · 예측 · 채점</td><td>1-1 <code>clf.fit(X_full, y_full)</code> · <code>clf.predict(X_full[:5])</code></td></tr>
<tr><td><code>accuracy_score()</code></td><td>sklearn.metrics</td><td>정답 비율</td><td>1-1 <code>accuracy_score(y_full, pred_all)</code></td></tr>
<tr><td><code>load_iris()</code></td><td>sklearn.datasets</td><td>붓꽃 데이터</td><td>0 <code>iris = load_iris()</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>np.random.default_rng().permutation()</code></td><td>numpy.random</td><td>무작위로 섞은 사본</td><td>1-2 <code>y_shuffled = rng.permutation(y_full)</code></td></tr>
<tr><td><code>train_test_split()</code></td><td>sklearn.model_selection</td><td>train · test 분할</td><td>2-1 <code>X_train, X_test, y_train, y_test = train_test_split(</code></td></tr>
<tr><td><code>test_size</code> · <code>stratify</code> · <code>random_state</code></td><td>train_test_split 인자</td><td>비율 · 클래스 비율 유지 · 분할 고정</td><td>2-1 <code>train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, …)</code></td></tr>
<tr><td><code>np.bincount()</code></td><td>numpy</td><td>정수 라벨별 개수</td><td>2-1 <code>np.bincount(y_test)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>KFold</code> · <code>StratifiedKFold</code></td><td>sklearn.model_selection</td><td>K등분 규칙 · 클래스 비율 유지 K등분</td><td>3-1 <code>skf = StratifiedKFold(n_splits=5, shuffle=True, ra…</code></td></tr>
<tr><td><code>.split(X, y)</code></td><td>분할기 메서드</td><td>(학습 번호, 검증 번호) K쌍</td><td>3-1 <code>folds = skf.split(X_train, y_train)</code></td></tr>
<tr><td><code>n_splits</code> · <code>shuffle</code> · <code>random_state</code></td><td>분할기 인자</td><td>등분 수 · 섞기 · 고정</td><td>3-1 <code>StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)</code></td></tr>
<tr><td><code>cross_val_score()</code></td><td>sklearn.model_selection</td><td>fold별 점수 배열</td><td>3-3 <code>scores = cross_val_score(tree, X_train, y_train, c…</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>cross_validate()</code></td><td>sklearn.model_selection</td><td>지표 여러 개 + 학습 점수 dict</td><td>3-4 <code>res = cross_validate(tree, X_train, y_train, cv=skf,</code></td></tr>
<tr><td><code>np.mean()</code> · <code>np.std()</code> · <code>np.allclose()</code></td><td>numpy</td><td>평균 · 표준편차 · 실수 비교</td><td>3-1 <code>np.mean(fold_scores)</code> · 3-3 <code>np.allclose(scores, fold_scores)</code></td></tr>
<tr><td><code>max_depth</code> · <code>min_samples_split</code></td><td>DecisionTreeClassifier 인자</td><td>트리 깊이 한도 · 나누는 최소 샘플 수</td><td>4-1 <code>DecisionTreeClassifier(max_depth=d, …)</code> · 4-3 <code>space</code></td></tr>
<tr><td><code>GridSearchCV()</code></td><td>sklearn.model_selection</td><td>모든 조합 교차검증 + refit</td><td>4-2 <code>grid = GridSearchCV(</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>RandomizedSearchCV()</code> · <code>n_iter</code></td><td>sklearn.model_selection</td><td>무작위 n_iter개 교차검증</td><td>4-3 <code>rand = RandomizedSearchCV(tree, dist, n_iter=20, c…</code></td></tr>
<tr><td><code>best_params_</code> · <code>best_score_</code> · <code>cv_results_</code></td><td>탐색기 속성</td><td>최적 설정 · 최적 CV 점수 · 전체 결과</td><td>4-2 <code>grid.best_params_</code> · <code>pd.DataFrame(grid.cv_results_)</code></td></tr>
<tr><td><code>randint()</code></td><td>scipy.stats</td><td>정수 균등 분포</td><td>4-3 <code>dist = {"max_depth": randint(1, 10), "min_samples_…</code></td></tr>
<tr><td><code>best_estimator_</code> · <code>refit=True</code></td><td>탐색기 속성 · 인자</td><td>최적 설정으로 train 전체에 다시 학습된 모델</td><td>5-1 <code>best_model = grid.best_estimator_</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>train_test_split()</code> 두 번</td><td>sklearn.model_selection</td><td>train · val · test 3분할</td><td>5-2 <code>X_rest, X_te3, …</code> → <code>X_tr3, X_val3, …</code></td></tr>
<tr><td><code>max(d, key=d.get)</code></td><td>Python 내장 함수</td><td>값이 가장 큰 키</td><td>5-2 <code>best_d = max(val_scores, key=val_scores.get)</code></td></tr>
<tr><td><code>max_features</code></td><td>DecisionTreeClassifier 인자</td><td>나눌 때 보는 특성 수</td><td>5-3 <code>DecisionTreeClassifier(max_depth=3, max_features=2, …)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>test는 맨 처음 떼어 봉인</td><td>학습한 데이터 점수를 실력으로 보고</td></tr>
<tr><td>분류는 <code>stratify</code> · <code>StratifiedKFold</code></td><td>정렬된 데이터를 섞지 않고 <code>KFold</code></td></tr>
<tr><td>모든 비교에 같은 분할기 · seed</td><td>모델마다 다른 분할로 비교</td></tr>
<tr><td>교차검증은 train 안에서만</td><td>전체 X로 교차검증 (test까지 섞임)</td></tr>
<tr><td>동점이면 단순한 설정</td><td>test 점수를 보고 설정을 다시 고치기</td></tr>
<tr><td><code>return_train_score=True</code>로 과적합 확인</td><td>test로 여러 번 채점해 최고점 고르기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>score</code> vs <code>accuracy_score</code></td><td>predict + 채점 vs 채점만</td><td>같은 값, 지표 여러 개면 <code>accuracy_score</code></td></tr>
<tr><td>학습 점수 vs test 점수</td><td>외운 문제 vs 처음 보는 문제</td><td>실력은 test 쪽</td></tr>
<tr><td><code>stratify</code> O vs X</td><td>클래스 비율 유지 vs 무작위</td><td>분류는 O</td></tr>
<tr><td><code>train_test_split</code> vs <code>KFold</code> vs <code>StratifiedKFold</code></td><td>1번 vs K번 vs K번 + 비율 유지</td><td>분류 검증은 <code>StratifiedKFold</code></td></tr>
<tr><td><code>KFold()</code> vs <code>KFold(shuffle=True)</code></td><td>순서대로 자름 vs 섞어서 자름</td><td>정렬된 데이터는 섞기</td></tr>
<tr><td><code>cv=5</code> vs <code>cv=StratifiedKFold(...)</code></td><td>섞지 않음 vs 섞기 · seed 지정</td><td>비교 실험은 객체를 넘김</td></tr>
<tr><td><code>cross_val_score</code> vs <code>cross_validate</code></td><td>지표 1개 vs 여러 개 + 학습 점수</td><td>과적합 확인은 <code>cross_validate</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>CV mean vs CV std</td><td>평균 실력 vs 흔들림</td><td>mean 비슷하면 std</td></tr>
<tr><td>과소적합 vs 과적합</td><td>둘 다 낮음 vs 학습만 높음</td><td>학습 − 검증 차이로 판단</td></tr>
<tr><td><code>GridSearchCV</code> vs <code>RandomizedSearchCV</code></td><td>모든 조합 vs n_iter개 무작위</td><td>범위가 넓으면 Random</td></tr>
<tr><td><code>best_score_</code> vs <code>best_estimator_</code></td><td>CV 점수 vs 다시 학습된 모델</td><td>test 채점은 <code>best_estimator_</code></td></tr>
<tr><td>cv 결과의 <code>test_</code> vs 최종 test</td><td>각 fold의 검증 점수 vs 봉인한 test 점수</td><td>이름만 같고 다른 것</td></tr>
<tr><td>검증(val) vs 최종 테스트(test)</td><td>여러 번 보며 고름 vs 마지막 1번</td><td>test로는 고르지 않음</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개를 새로 풀기 · 데이터 분할 흐름도를 가리고 직접 그리기</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>6장 펭귄 실습을 새 데이터(와인 <code>load_wine</code>)로 처음부터 다시 짜기</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 다시 풀기 · 머신러닝-2 시작 전 2 · 3 · 5장 ✅ 줄만 읽기</td><td>15분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 머신러닝 수업 노트북 2개(intro · model_selection)를 **중복 없이 5개 대단원 · 15개 섹션**으로 재구성하고, **6장 종합 실습**(펭귄)에서 하나로 연결.
- model_selection의 **빈칸 32칸은 정답을 직접 작성해 실행으로 검증**했고, 이후 머신러닝-2(전처리 · 파이프라인)
- 머신러닝-3(평가지표)과 딥러닝의 train · val
- test 흐름을 읽기 위한 기초.

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 예측 정답 · 결과 해설 → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 실행 흐름 · 문법 구조

**셀 실행** · 위에서부터 차례로 (앞 섹션의 변수를 이어서 씀), 모든 분할 · 모델은 `random_state` 고정 → 같은 환경에서 다시 실행하면 같은 숫자

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">섹션</th></tr>
</thead>
<tbody>
<tr><td>01 intro</td><td>머신러닝 정의 · 종류 · 용어 · <code>fit → predict → score</code> · 환경 버전</td><td>0 (버전) · 1장 첫머리 (종류 · 용어 표) · 1-1</td></tr>
<tr><td>02 셀 0 ~ 7 · <strong>빈칸 3 ~ 5</strong></td><td>전체로 학습 · 채점 → 1.000 착시</td><td>1-2 (빈칸 = 다른 모델 · 엉터리 정답으로 착시 확인)</td></tr>
<tr><td>02 셀 8 ~ 10 · <strong>빈칸 11 ~ 22</strong></td><td>train / test split</td><td>2-1 · 2-2 (빈칸 = stratify 비교 · 분할 seed별 점수 흔들림)</td></tr>
<tr><td>02 Part 2 셀 24 ~ 28</td><td>K-Fold 직접 구현</td><td>3-1</td></tr>
<tr><td>02 <strong>빈칸 29 ~ 36</strong></td><td>(K-Fold 구현 뒤 실험)</td><td>3-2 (KFold vs StratifiedKFold · shuffle)</td></tr>
<tr><td>02 셀 37 · <strong>빈칸 38 ~ 43</strong> · 셀 44 ~ 46</td><td>"for문은 귀찮" → cross_val_score</td><td>3-3 · 3-4 (cross_validate)</td></tr>
<tr><td>02 셀 47 ~ 54 · <strong>빈칸 50 · 51</strong></td><td>모델 비교 · mean과 std · 선택 규칙</td><td>3-5 (빈칸 = 분할 seed를 바꿔 순위 재확인)</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">섹션</th></tr>
</thead>
<tbody>
<tr><td>02 Part 3 셀 55 ~ 62</td><td>max_depth · GridSearchCV · 오해</td><td>4-1 · 4-2 · 4-3 (셀 27 · 58 그래프는 같은 숫자를 표로 출력)</td></tr>
<tr><td>02 Part 4 셀 63 ~ 68</td><td>최종 홀드아웃 · 정리</td><td>5-1 (5-2 · 5-3은 딥러닝 연결로 추가)</td></tr>
<tr><td>02 실습 미션 1 ~ 4 · <strong>빈칸 72</strong></td><td>max_depth · KNN · 결과표 · Grid 확장</td><td>7-2 코드 문제 (빈칸 72 = 미션 3 결과표)</td></tr>
<tr><td>2-1 펭귄 실습 문제 · 정답</td><td>같은 흐름의 연습</td><td>6장 종합 실습 재료로만 사용</td></tr>
</tbody></table>

빈칸 32칸은 출력도 남아 있지 않아, **앞뒤 설명 셀이 묻는 내용**으로 들어갈 코드를 정함. 원본 파일은 수정하지 않음.

### 원본에서 고친 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본 위치</th><th style="text-align: left;">문제</th><th style="text-align: left;">정리본</th></tr>
</thead>
<tbody>
<tr><td>셀 26</td><td>전체 X로 교차검증 (test까지 섞임)</td><td>test를 봉인하고 X_train으로 (3-1)</td></tr>
<tr><td>셀 46</td><td>주석은 5-Fold, 코드는 <code>cv=4</code>, 저장된 출력은 7개</td><td><code>cv=skf</code> 5등분으로 통일 (3-3)</td></tr>
<tr><td>셀 60 · 61</td><td><code>return_train_score=True</code> 주석 → <code>mean_train_score</code>에서 <code>KeyError</code></td><td>켜서 실행 (4-2)</td></tr>
<tr><td>셀 54</td><td><code>"\상위 2개…"</code> 역슬래시 오타</td><td>선택 규칙만 짧게 (3-5)</td></tr>
<tr><td>원본 버전</td><td>sklearn 1.6.1</td><td>이 노트북은 실행 환경 버전으로 다시 계산 (0장 출력), 숫자는 원본 출력과 다를 수 있음</td></tr>
</tbody></table>

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">순서</th><th style="text-align: left;">배우는 이유</th><th style="text-align: left;">대단원</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">1</td><td>모델 사용법을 익히고, 그 점수가 왜 가짜인지 확인한다</td><td>왜 나누는가</td></tr>
<tr><td style="text-align: center;">2</td><td>처음 보는 데이터를 만들려고 test를 떼어 둔다</td><td>데이터 나누기</td></tr>
<tr><td style="text-align: center;">3</td><td>한 번 나눈 점수는 운이 섞여서 여러 번 나눠 평균을 낸다</td><td>교차검증</td></tr>
<tr><td style="text-align: center;">4</td><td>교차검증 점수로 설정값을 공정하게 고른다</td><td>하이퍼파라미터 탐색</td></tr>
<tr><td style="text-align: center;">5</td><td>고른 모델을 봉인한 test로 한 번만 확인한다 → 딥러닝의 3분할</td><td>최종 테스트 · 딥러닝 연결</td></tr>
<tr><td style="text-align: center;">6</td><td>1~5를 새 데이터(펭귄)에 처음부터 끝까지 적용한다</td><td>종합 실습</td></tr>
</tbody></table>

#### ML/DL 공통 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰이는 장</th></tr>
</thead>
<tbody>
<tr><td>① 데이터 준비</td><td>데이터 읽기 · 결측 제거</td><td>0장 · 6장</td></tr>
<tr><td>② 나누기</td><td>train · val · test 분할, K-Fold</td><td><strong>2장</strong> · <strong>3장</strong> · <strong>5장</strong></td></tr>
<tr><td>③ 전처리</td><td>스케일링 · 인코딩 (train으로만 fit)</td><td>머신러닝-2</td></tr>
<tr><td>④ 모델</td><td>모델 만들기 · 설정값</td><td>1장 · 4장</td></tr>
<tr><td>⑤ 학습</td><td><code>fit</code> / 학습 루프</td><td>1장</td></tr>
<tr><td>⑥ 평가 · 선택</td><td>검증 점수로 모델 · 설정 고르기</td><td><strong>3장</strong> · <strong>4장</strong></td></tr>
<tr><td>⑦ 최종 보고</td><td>test 1회</td><td><strong>5장</strong></td></tr>
</tbody></table>

**굵은 장** · 이 노트북의 핵심 = ② 나누기와 ⑥ 고르기, ⑦ 최종 보고의 경계

### 전체 코드

1 ~ 6장 핵심 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 출력 앞 `[번호]`가 섹션.

**실행 흐름**

1. 전체 코드가 쓰는 도구 불러오기
2. seed · iris 데이터 준비

In [21]:
# ---------- 0. 준비 ----------
# 왜 · 이 셀들만 위에서부터 실행해도 전체가 동작하도록 도구 · 데이터를 먼저 준비하기 위해
import numpy as np                                   # 배열 · 평균
import pandas as pd                                  # 결과 표
import seaborn as sns                                # 펭귄 데이터 (6장)
from scipy.stats import randint                      # 정수 분포 (4-3)
from sklearn.datasets import load_iris               # 붓꽃 데이터
from sklearn.tree import DecisionTreeClassifier      # 트리 모델
from sklearn.linear_model import LogisticRegression  # 로지스틱 모델
from sklearn.neighbors import KNeighborsClassifier   # KNN 모델
from sklearn.metrics import accuracy_score           # 정확도
from sklearn.model_selection import (                # 나누기 · 교차검증 · 탐색 도구
    train_test_split, KFold, StratifiedKFold, cross_val_score,
    cross_validate, GridSearchCV, RandomizedSearchCV)

SEED = 18                                         # 원본과 같은 seed → 같은 환경에서 같은 결과
iris = load_iris()                                # sklearn 안에 든 데이터 → 파일 필요 없음
X_full, y_full = iris.data, iris.target           # 문제 · 정답
tree = DecisionTreeClassifier(random_state=SEED)  # 학습 전 모델을 넘김 (안에서 복사)
print("[0]", X_full.shape, np.bincount(y_full))   # 모양 · 클래스별 개수

[0] (150, 4) [50 50 50]


**실행 흐름**

1. 1-1 fit → predict → score
2. 1-2 같은 데이터 채점의 착시

In [22]:
# ---------- 1. 왜 나누는가 ----------
# 1-1 fit → predict → score
# 왜 · 모델 사용의 기본 4단계를 다시 확인하기 위해
clf = DecisionTreeClassifier(random_state=SEED).fit(X_full, y_full)  # fit은 모델 자신을 돌려줌
acc = accuracy_score(y_full, clf.predict(X_full))                    # 채점만 하는 방법
print("[1-1]", clf.score(X_full, y_full) == acc)                     # 두 방법이 같은 값

# 1-2 학습한 데이터로 채점하면 엉터리 정답도 거의 100%
y_shuffled = np.random.default_rng(SEED).permutation(y_full)              # 엉터리 정답
fake = DecisionTreeClassifier(random_state=SEED).fit(X_full, y_shuffled)  # 엉터리 정답으로 학습
print("[1-2]", round(fake.score(X_full, y_shuffled), 3))                  # 그래도 0.993

[1-1] True
[1-2] 0.993


**실행 흐름**

1. 2-1 8 : 2 분할 (stratify)
2. 2-2 분할 seed에 따른 점수 흔들림

In [23]:
# ---------- 2. 데이터 나누기 ----------
# 2-1 8 : 2 분할 (stratify)
# 왜 · 클래스 비율을 지키며 test를 떼어 내기 위해
X_train, X_test, y_train, y_test = train_test_split(              # 8 : 2로 나눔
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=SEED)
print("[2-1]", X_train.shape, X_test.shape, np.bincount(y_test))  # 120 · 30 · 비율 유지

# 2-2 분할 seed만 바꿔도 test 점수가 흔들림
split_scores = []                                            # seed별 test 점수를 모을 곳
for rs in range(10):                                         # 분할 seed 0 ~ 9
    Xtr, Xte, ytr, yte = train_test_split(                   # 시험지만 바꿔 나누기
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=rs)
    split_scores.append(tree.fit(Xtr, ytr).score(Xte, yte))  # 같은 모델 · 다른 시험지
print("[2-2]", round(min(split_scores), 3), round(max(split_scores), 3))  # 최저 · 최고

[2-1] (120, 4) (30, 4) [10 10 10]
[2-2] 0.9 0.967


**실행 흐름**

1. 3-1 StratifiedKFold 직접 반복
2. 3-2 섞지 않은 KFold의 위험
3. 3-3 · 3-4 `cross_val_score` · 모델 비교

In [24]:
# ---------- 3. 교차검증 ----------
# 3-1 StratifiedKFold 직접 반복
# 왜 · 교차검증이 하는 일을 다시 확인하기 위해
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # 규칙만 만듦 (아직 안 나눔)
fold_scores = []                                                    # fold별 점수를 모을 곳
for t, v in skf.split(X_train, y_train):                            # (학습 번호, 검증 번호) 5쌍
    tree.fit(X_train[t], y_train[t])                                # 4등분으로 학습
    fold_scores.append(tree.score(X_train[v], y_train[v]))          # 1등분으로 채점
print("[3-1]", round(np.mean(fold_scores), 3), round(np.std(fold_scores), 3))  # 평균 · 흔들림

# 3-2 정렬된 데이터를 섞지 않고 KFold → 0점 fold
accs = []                                                                    # fold별 정확도
for t, v in KFold(n_splits=3).split(X_full):                                 # 섞지 않고 3등분
    accs.append(tree.fit(X_full[t], y_full[t]).score(X_full[v], y_full[v]))  # 검증 fold 정확도
print("[3-2]", np.round(accs, 3))                                            # 전부 0점

# 3-3 cross_val_score = 3-1 for문
scores = cross_val_score(tree, X_train, y_train, cv=skf)  # 3-1을 한 줄로
print("[3-3]", np.allclose(scores, fold_scores))          # 같은 값인지

# 3-4 cross_validate → 지표 여러 개 + 학습 점수
res = cross_validate(tree, X_train, y_train, cv=skf,    # 같은 5등분
                     scoring=["accuracy", "f1_macro"],
                     return_train_score=True)
print("[3-4]", round(res["train_accuracy"].mean(), 3),  # 학습 · 검증 평균
      round(res["test_accuracy"].mean(), 3))

# 3-5 같은 5등분으로 모델 3개 비교
models = {"DecisionTree": DecisionTreeClassifier(max_depth=5,         # 비교할 모델 3개
                                                 random_state=SEED),
          "LogisticRegression": LogisticRegression(max_iter=2000),
          "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)}
cv_means = {}                                                         # 모델별 CV 평균
for n, m in models.items():                                           # 모델마다
    cv_means[n] = float(cross_val_score(m, X_train, y_train, cv=skf)  # 같은 5등분
                        .mean().round(3))
print("[3-5]", cv_means)                                              # 평균 비교

[3-1] 0.958 0.037
[3-2] [0. 0. 0.]
[3-3] True
[3-4] 1.0 0.958


[3-5] {'DecisionTree': 0.958, 'LogisticRegression': 0.967, 'KNN (k=5)': 0.983}


**실행 흐름**

1. 4-1 · 4-2 GridSearchCV
2. 4-3 RandomizedSearchCV

In [25]:
# ---------- 4. 하이퍼파라미터 탐색 ----------
# 4-1 · 4-2 GridSearchCV (학습 점수도 기록)
# 왜 · 후보 탐색과 과적합 정도를 함께 보기 위해
grid = GridSearchCV(tree, {"max_depth": [2, 3, 4, None]}, cv=skf,
                    return_train_score=True).fit(X_train, y_train)
print("[4-2]", grid.best_params_, round(grid.best_score_, 3))  # 고른 설정 · CV 점수

# 4-3 RandomizedSearchCV → 넓은 범위에서 20개만
dist = {"max_depth": randint(1, 10), "min_samples_split": randint(2, 20)}  # 같은 범위의 분포
rand = RandomizedSearchCV(tree, dist, n_iter=20, cv=skf,                   # 20가지만 뽑음
                          random_state=SEED).fit(X_train, y_train)
print("[4-3]", rand.best_params_, round(rand.best_score_, 3))              # 20개만으로 같은 점수

[4-2] {'max_depth': 2} 0.958


[4-3] {'max_depth': 2, 'min_samples_split': 16} 0.958


**실행 흐름**

1. 5-1 test 1회
2. 5-2 3분할 · val로 고르기

In [26]:
# ---------- 5. 최종 테스트 · 딥러닝 연결 ----------
# 5-1 test 1회
# 왜 · 선택이 끝난 뒤 test를 한 번만 쓰기 위해
print("[5-1]", round(grid.best_estimator_.score(X_test, y_test), 3))  # test 1회

# 5-2 train · val · test 3분할 → val로 고르고 test 1회
X_rest, X_te3, y_rest, y_te3 = train_test_split(                     # test 20% 먼저 봉인
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=SEED)
X_tr3, X_val3, y_tr3, y_val3 = train_test_split(                     # 남은 80%의 25% = 전체 20%
    X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=SEED)
val_scores = {}                                                      # 깊이별 val 점수
for d in [1, 2, 3, 4, None]:                                         # 깊이 후보마다
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)       # 깊이 후보
    val_scores[d] = m.fit(X_tr3, y_tr3).score(X_val3, y_val3)        # train 학습 → val 채점
best_d = max(val_scores, key=val_scores.get)                         # val 최고 깊이
final = DecisionTreeClassifier(max_depth=best_d, random_state=SEED)  # 고른 설정으로
print("[5-2]", best_d, round(final.fit(X_tr3, y_tr3).score(X_te3, y_te3), 3))  # test 1회

[5-1] 0.9
[5-2] 2 0.9


**실행 흐름**

1. 펭귄 데이터 준비 · test 봉인
2. train 안 5-Fold 비교 · 깊이 탐색
3. test 1회

In [27]:
# ---------- 6. 종합 실습: 펭귄 ----------
# 왜 · 펭귄 데이터로 분할 → CV → 탐색 → test 1회 흐름을 다시 실행하기 위해
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",    # 측정값 4열
        "body_mass_g"]
peng = sns.load_dataset("penguins").dropna(subset=meas)            # 결측 2마리 제거
X_p, y_p = peng[meas].to_numpy(), peng["species"].to_numpy()       # 표 → 배열
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(                     # test 20% 봉인
    X_p, y_p, test_size=0.2, stratify=y_p, random_state=7)
cv_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 비교용 5등분
grid_p = GridSearchCV(DecisionTreeClassifier(random_state=42),     # 깊이 탐색
                      {"max_depth": [1, 2, 3, 4, None]}, cv=cv_p)
grid_p.fit(Xp_tr, yp_tr)                                           # train 안에서만
print("[6]", grid_p.best_params_, round(grid_p.best_score_, 3),    # 최적 · CV · test
      round(grid_p.score(Xp_te, yp_te), 3))

[6] {'max_depth': 4} 0.963 0.913
